# Cross-domain AF detection

Research pipeline for an AF-versus-explicit-sinus-rhythm study across PTB-XL, Shaoxing and Georgia. Dataset downloads and experiment outputs are generated when the pipeline runs; they are not included in this repository. Read the project README before running it.


In [ ]:
# ============================================================================
# Notebook cell 1 (zero-based 0): markdown
# ============================================================================
# # Cross-domain AF detection: executable Colab experiment
#
# **Input:** numerical 12-lead ECG. **Task:** explicit atrial fibrillation (AF) versus explicit sinus rhythm (SR). **Output:** trained checkpoints, portable inference bundle, statistical tables, figures, predictions and a downloadable ZIP.
#
# Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Drive mounting requires your consent in Colab. No data upload is needed. The default `pilot` mode targets roughly 1–2 hours on a T4 **under favorable download speeds**; this is a budget, not a measured guarantee. The full predeclared study can take substantially longer. Neither zero errors nor constant maximum GPU utilization can be guaranteed. Downloads, CPU preprocessing, validation and export necessarily leave the GPU idle.
#
# | Setting | Pilot (default) | Full research |
# |---|---|---|
# | External domain | Georgia | Georgia, Shaoxing, PTB-XL |
# | Training records per source | All eligible source-training records | All eligible |
# | Seeds for selected ResNet method | 42, 202, 3407 | 42, 202, 3407 |
# | Alignment weights | 0, .01, .05 | 0, .01, .05, .1 |
# | Maximum epochs / patience | 25 / 6 | 40 / 8 |
# | Per-fit training time cap | 5 minutes | None |
# | Patient bootstrap repetitions | 200 | 1,000 |
#
# This revision is exploratory because the previous Georgia external results were inspected. A new independent held-out cohort is required for confirmatory comparisons. Pilot results are feasibility results, not completion of the full research plan. All evaluation sets retain their natural class prevalence. The model is a research demonstration, not a diagnostic device. PTB-XL is treated as a **dataset domain**, not asserted to be one hospital.
#
# **Provenance change from the supplied plan:** use PTB-XL v1.0.3 plus ECG-Arrhythmia v1.0.0. Hospital membership is joined by waveform-file SHA-256 against the official Challenge 2021 v1.0.3 checksum manifest. This verifies membership without guessing from record number ranges; any unmatched or ambiguous origin stops the run. No Challenge PTB-XL copy is used. The two Chinese cohorts are documented as one recording per patient; their record identifiers are dataset-supported patient proxies, not independently linkable patient identities. Unknown cross-institution patient overlap cannot be ruled out.
#
# **Verification before delivery:** see the final notebook markdown for the tests actually performed. Full real-data/T4 training is not represented as already executed.

# ============================================================================
# Notebook cell 2 (zero-based 1): markdown
# ============================================================================
# ## 01–03 · Environment, persistent outputs and fixed configuration
# Raw data and caches use Colab local SSD; small research outputs are mirrored to Drive after each fit. Re-running uses a new run directory, preventing stale model reuse. Install only missing analysis dependencies; preserve Colab's CUDA-enabled PyTorch. Package versions are saved for reproducibility. For strict replication, use those saved versions in a fresh environment.

# ============================================================================
# Notebook cell 3 (zero-based 2): code
# ============================================================================
import sys, subprocess, importlib.util, time, os
START = time.monotonic()
required = {'wfdb':'wfdb==4.3.0', 'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy',
            'sklearn':'scikit-learn', 'matplotlib':'matplotlib', 'psutil':'psutil',
            'requests':'requests', 'tqdm':'tqdm'}
missing = [spec for mod,spec in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2,3), 'Use a current Colab runtime.'

# ============================================================================
# Notebook cell 4 (zero-based 3): code
# ============================================================================
from pathlib import Path
import json, ast, random, hashlib, shutil, gc, platform, warnings, math, re
import numpy as np
import pandas as pd
import scipy
from scipy.signal import resample_poly, butter, sosfiltfilt
from scipy.optimize import minimize_scalar
from scipy.special import expit
import matplotlib.pyplot as plt
import psutil, requests, wfdb
from tqdm.auto import tqdm
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
    f1_score, accuracy_score, brier_score_loss, log_loss, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
MODE = 'pilot'  # Exploratory protocol revision after inspecting the previous Georgia evaluation.
MOUNT_DRIVE = True
AUTO_DOWNLOAD = True
CFG = dict(mode=MODE, fs=100, seconds=10, seed=42, lr=1e-3, weight_decay=1e-4,
    max_batch=256, min_batch=32, workers=2, alignment=[0,.01,.05],
    epochs=25, patience=6, seeds=[42,202,3407], targets=['Georgia'],
    train_cap=None, fit_seconds=300, bootstrap=200, total_budget_minutes=115,
    download_workers=3, strict_gpu=True,
    operating_point='source_sensitivity', source_target_sensitivity=0.97,
    source_min_specificity=0.95, exploratory_revision=True)
if MODE == 'full':
    CFG.update(epochs=40, patience=8, seeds=[42,202,3407],
        targets=['Georgia','Shaoxing','PTBXL'], train_cap=None,
        alignment=[0,.01,.05,.1], fit_seconds=None, bootstrap=1000,
        total_budget_minutes=None)
assert MODE in ['pilot','full']
assert CFG['operating_point'] in ['source_sensitivity','youden']
ROOT = Path('/content/ecg_cross_domain') if Path('/content').exists() else Path.cwd()/'ecg_cross_domain'
RAW, CACHE = ROOT/'raw', ROOT/'cache'
RUN = time.strftime('%Y%m%d_%H%M%S')
OUT = ROOT/'runs'/RUN
for p in [RAW,CACHE,OUT/'models',OUT/'tables',OUT/'figures',OUT/'predictions',OUT/'bundle']:
    p.mkdir(parents=True,exist_ok=True)
DRIVE = None
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print('Outside Colab: persistent Drive mirror unavailable.')
    else:
        drive.mount('/content/drive')
        DRIVE = Path('/content/drive/MyDrive/ECG_CrossDomain')/RUN
        DRIVE.mkdir(parents=True,exist_ok=True)
def write_json(p,obj):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str)); tmp.replace(p)
def sync_outputs():
    if DRIVE is not None: shutil.copytree(OUT,DRIVE,dirs_exist_ok=True)
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda' or not CFG['strict_gpu'], 'Select a GPU runtime before downloading.'
torch.set_num_threads(min(4,os.cpu_count() or 2))
torch.backends.cudnn.benchmark=True  # Throughput; exact bitwise determinism is not claimed.
if DEVICE.type=='cuda':
    print(torch.cuda.get_device_name(0))
    assert torch.cuda.get_device_properties(0).total_memory >= 10*2**30, 'At least 10 GiB VRAM required.'
assert psutil.virtual_memory().available >= 4*2**30, 'Need >=4 GiB free system RAM.'
assert shutil.disk_usage(ROOT).free >= 22*2**30, 'Need >=22 GiB free local disk.'
versions=dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,
    numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,wfdb=wfdb.__version__)
write_json(OUT/'config.json',CFG); write_json(OUT/'environment.json',versions)
(OUT/'requirements-lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(CFG); print('Outputs:',OUT)
print('Initialization finished. Run the next code cell, Section 04, to start downloading.', flush=True)

# ============================================================================
# Notebook cell 5 (zero-based 4): markdown
# ============================================================================
# ## 04 · Fast official dataset downloads with Boto3 and targeted Georgia HTTPS
#
# PTB-XL v1.0.3 and ECG Arrhythmia v1.0.0 are retrieved from PhysioNet’s anonymous public Amazon S3 bucket using the supplied Boto3 implementation: 8 parallel workers per dataset, concurrent dataset downloads, retry/resume, and 100 Hz-only PTB-XL selection. SHA-256 checks and the Chinese hospital-origin manifest are retained. The Arrhythmia mirror contains both Shaoxing and Ningbo recordings; Ningbo remains excluded from the AFIB-versus-SR model because it lacks AFIB labels. Georgia headers are independently fetched from the official Challenge 2021 HTTPS release and verified with its SHA-256 manifest. After AFIB/SR and 10-second eligibility preflight, only qualifying Georgia waveforms are fetched, also using concurrent resumable requests. This mixed transport is not a benchmarked fastest option; the original Boto3 S3 approach is retained where its public S3 paths are documented. Download failures halt the experiment.

# ============================================================================
# Notebook cell 6 (zero-based 5): code
# ============================================================================

# ============================================================
# 04 · OFFICIAL ECG DOWNLOADS
# Google Colab | Boto3 S3 | ZIP fallback
# ============================================================

import importlib.util
import sys
import subprocess
import time
import tarfile
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Event, Lock, Thread

# ============================================================
# DOWNLOAD CONFIGURATION
# ============================================================

print('[DOWNLOAD CELL] Started; checking selected dataset sources.', flush=True)
assert CFG['targets'][0] == 'Georgia', (
    'Old notebook/configuration active: expected Georgia target, got ' + str(CFG['targets']) +
    '. Restart runtime and run this updated notebook from the top.'
)
DOWNLOAD_BACKEND = 'aws_s3'       # 'aws_s3' or 'zip'
PTB_100HZ_ONLY = True           # Skip unnecessary 500 Hz files
S3_WORKERS_PER_DATASET = 8      # Up to 16 requests across 2 datasets
S3_REPORT_EVERY_SECONDS = 20

# ============================================================
# OFFICIAL PHYSIONET DATA SOURCES
# ============================================================

SOURCES = {
    'PTBXL': {
        'url': (
            'https://physionet.org/static/published-projects/'
            'ptb-xl/ptb-xl-a-large-publicly-available-'
            'electrocardiography-dataset-1.0.3.zip'
        ),
        'name': 'ptbxl.zip',
        's3': 's3://physionet-open/ptb-xl/1.0.3/'
    },
    'Arrhythmia': {
        'url': (
            'https://physionet.org/static/published-projects/'
            'ecg-arrhythmia/a-large-scale-12-lead-'
            'electrocardiogram-database-for-arrhythmia-'
            'study-1.0.0.zip'
        ),
        'name': 'arrhythmia.zip',
        's3': 's3://physionet-open/ecg-arrhythmia/1.0.0/'
    }
}

# ============================================================
# CHECKSUM HELPER
# ============================================================

def digest(path, algorithm='sha256'):
    h = hashlib.new(algorithm)

    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(2**20), b''):
            h.update(block)

    return h.hexdigest()


# ============================================================
# RESUMABLE HTTP DOWNLOADER
# Used for ZIP fallback and small checksum manifests
# ============================================================

def download(url, dest, md5=None):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    part = Path(str(dest) + '.part')

    if dest.is_file():
        if md5 is None or digest(dest, 'md5') == md5:
            return dest
        dest.unlink()

    last_error = None

    for attempt in range(5):
        try:
            offset = part.stat().st_size if part.exists() else 0

            headers = {'Accept-Encoding': 'identity'}

            if offset:
                headers['Range'] = f'bytes={offset}-'

            with requests.get(
                url,
                stream=True,
                timeout=(30, 120),
                headers=headers
            ) as response:

                if response.status_code == 416:
                    part.unlink(missing_ok=True)
                    continue

                response.raise_for_status()

                append = (
                    offset > 0
                    and response.status_code == 206
                )

                if append:
                    content_range = response.headers.get(
                        'Content-Range', ''
                    )

                    if not content_range.startswith(
                        f'bytes {offset}-'
                    ):
                        raise IOError(
                            f'Unexpected HTTP Content-Range: '
                            f'{content_range}'
                        )

                length = response.headers.get('Content-Length')

                expected = (
                    int(length)
                    if length and length.isdigit()
                    else None
                )

                received = 0
                report_at = time.monotonic()

                with open(
                    part,
                    'ab' if append else 'wb'
                ) as output:

                    for chunk in response.iter_content(
                        chunk_size=2**20
                    ):
                        if chunk:
                            output.write(chunk)
                            received += len(chunk)

                        if time.monotonic() - report_at >= 20:
                            print(
                                f'[HTTP {dest.name}] '
                                f'{part.stat().st_size / 2**20:,.1f} MiB',
                                flush=True
                            )
                            report_at = time.monotonic()

                if expected is not None and received != expected:
                    raise IOError(
                        f'Incomplete HTTP response: '
                        f'{received}/{expected}'
                    )

            if md5 is not None and digest(part, 'md5') != md5:
                part.unlink(missing_ok=True)
                raise ValueError('ZIP MD5 checksum mismatch')

            part.replace(dest)

            return dest

        except (
            requests.RequestException,
            OSError,
            ValueError
        ) as exc:
            last_error = exc

            print(
                f'[HTTP] Retry {attempt + 1}/5 '
                f'for {dest.name}: {exc}',
                flush=True
            )

            time.sleep(min(2**attempt, 16))

    raise RuntimeError(
        f'HTTP download failed for {url}: {last_error}'
    )


# ============================================================
# SAFE ZIP EXTRACTION FALLBACK
# ============================================================

def extract_safe(archive, dest, ptb_only=False):
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)

    marker = dest / 'extraction_complete.json'
    sha = digest(archive)

    marker_data = {
        'sha256': sha,
        'ptb_only': bool(ptb_only)
    }

    if marker.exists():
        try:
            if json.loads(marker.read_text()) == marker_data:
                return
        except (ValueError, OSError):
            pass

    def check_path(name):
        if not (dest / name).resolve().is_relative_to(
            dest.resolve()
        ):
            raise ValueError(f'Unsafe archive path: {name}')

    def selected(name):
        if not ptb_only:
            return True

        parts = Path(name).parts

        return (
            'records100' in parts
            or Path(name).name in {
                'ptbxl_database.csv',
                'scp_statements.csv',
                'SHA256SUMS.txt',
                'LICENSE.txt'
            }
        )

    if str(archive).lower().endswith('.zip'):
        with zipfile.ZipFile(archive) as z:
            items = [
                i for i in z.infolist()
                if selected(i.filename)
            ]

            if ptb_only and not any(
                'records100' in Path(i.filename).parts
                and not i.is_dir()
                for i in items
            ):
                raise ValueError(
                    'PTB-XL ZIP has no records100 files'
                )

            for i in items:
                check_path(i.filename)

                if (
                    (i.external_attr >> 16) & 0o170000
                ) == 0o120000:
                    raise ValueError(
                        f'Archive symbolic link: {i.filename}'
                    )

            for i in tqdm(
                items,
                desc=f'Extracting {Path(archive).name}',
                leave=False
            ):
                z.extract(i, dest)

    else:
        with tarfile.open(archive, 'r:gz') as t:
            members = [
                m for m in t.getmembers()
                if selected(m.name)
            ]

            for m in members:
                check_path(m.name)

                if not (m.isfile() or m.isdir()):
                    raise ValueError(
                        f'Archive link/device: {m.name}'
                    )

            t.extractall(
                dest,
                members=members,
                filter='data'
            )

    write_json(marker, marker_data)


# ============================================================
# BOTO3 INSTALLATION
# No AWS CLI, no docutils, no standalone binary
# ============================================================

def ensure_boto3():
    if importlib.util.find_spec('boto3') is None:
        print(
            '[S3] Installing boto3 (no AWS CLI required)...',
            flush=True
        )

        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            'boto3'
        ])

    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config

    print(
        f'[S3] boto3 {boto3.__version__}; '
        'anonymous public S3 access',
        flush=True
    )

    return boto3, UNSIGNED, Config


# ============================================================
# OFFICIAL S3 PARALLEL DOWNLOADER
# ============================================================

def sync_from_official_s3(domain, spec):
    """
    Features:
      - Public PhysioNet AWS S3
      - No AWS credentials
      - Parallel downloads
      - Per-object byte-range continuation on retry
      - Reuse previously downloaded complete-size files
      - Live transfer progress
      - PTB-XL 100 Hz-only selection

    Published SHA-256 checks are retained downstream.
    """

    boto3, unsigned, aws_config = ensure_boto3()

    bucket, prefix = spec['s3'][5:].split('/', 1)

    dest = RAW / domain
    dest.mkdir(parents=True, exist_ok=True)

    workers = max(
        1,
        int(S3_WORKERS_PER_DATASET)
    )

    # --------------------------------------------------------
    # Anonymous public S3 client
    # --------------------------------------------------------

    s3 = boto3.client(
        's3',
        region_name='us-east-1',
        config=aws_config(
            signature_version=unsigned,
            connect_timeout=30,
            read_timeout=120,
            max_pool_connections=workers + 4,
            retries={
                'mode': 'standard',
                'max_attempts': 5
            }
        )
    )

    # --------------------------------------------------------
    # List required objects
    # --------------------------------------------------------

    if domain == 'PTBXL' and PTB_100HZ_ONLY:
        subtrees = [
            prefix + 'records100/'
        ]
    else:
        subtrees = [prefix]

    objects = {}

    print(
        f'[S3 {domain}] Listing {spec["s3"]}; counts update every 2 pages.',
        flush=True
    )

    paginator = s3.get_paginator('list_objects_v2')

    pages = 0

    for sub in subtrees:
        for page in paginator.paginate(
            Bucket=bucket,
            Prefix=sub
        ):
            pages += 1

            for entry in page.get('Contents', []):
                if not entry['Key'].endswith('/'):
                    objects[entry['Key']] = int(entry['Size'])

            if pages == 1 or pages % 2 == 0:
                print(
                    f'[S3 {domain}] Discovered '
                    f'{len(objects):,} files...',
                    flush=True
                )

    # --------------------------------------------------------
    # Include necessary PTB-XL metadata
    # --------------------------------------------------------

    if domain == 'PTBXL' and PTB_100HZ_ONLY:
        for filename in (
            'ptbxl_database.csv',
            'scp_statements.csv',
            'SHA256SUMS.txt',
            'LICENSE.txt'
        ):
            key = prefix + filename

            metadata = s3.head_object(
                Bucket=bucket,
                Key=key
            )

            objects[key] = int(metadata['ContentLength'])

    if not objects:
        raise RuntimeError(
            f'[S3 {domain}] No files found. '
            'Verify Colab Internet access.'
        )

    # --------------------------------------------------------
    # Reuse completed files
    # --------------------------------------------------------

    base = dest.resolve()

    missing = []

    kept_files = 0
    kept_bytes = 0

    for key, expected_size in objects.items():
        rel = key[len(prefix):]

        target = (dest / rel).resolve()

        if not target.is_relative_to(base):
            raise ValueError(
                f'Unsafe S3 key: {key}'
            )

        if (
            target.is_file()
            and target.stat().st_size == expected_size
        ):
            kept_files += 1
            kept_bytes += expected_size
        else:
            missing.append((
                key,
                expected_size,
                target
            ))

    total_bytes = sum(objects.values())

    print(
        f'[S3 {domain}] '
        f'{len(objects):,} remote files | '
        f'{total_bytes / 2**20:,.0f} MiB total',
        flush=True
    )

    print(
        f'[S3 {domain}] '
        f'{kept_files:,} existing complete-size files | '
        f'{len(missing):,} files to download | '
        f'{workers} workers',
        flush=True
    )

    # --------------------------------------------------------
    # Progress tracking
    # --------------------------------------------------------

    lock = Lock()
    stop = Event()

    downloaded = 0
    completed = 0
    complete_bytes = 0

    started = time.monotonic()

    def reporter():
        while not stop.wait(
            max(5, int(S3_REPORT_EVERY_SECONDS))
        ):
            with lock:
                n = completed
                completed_size = complete_bytes
                transferred = downloaded

            seconds = max(
                1,
                time.monotonic() - started
            )

            print(
                f'[S3 {domain}] '
                f'{seconds / 60:.1f} min | '
                f'{kept_files + n:,}/{len(objects):,} files done | '
                f'{(kept_bytes + completed_size) / 2**20:,.1f}/'
                f'{total_bytes / 2**20:,.1f} MiB complete | '
                f'{transferred / 2**20 / seconds:.2f} MiB/s received',
                flush=True
            )

    # --------------------------------------------------------
    # Download each object with retry and resume
    # --------------------------------------------------------

    def get_one(item):
        nonlocal downloaded, completed, complete_bytes

        key, size, target = item

        target.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        part = Path(
            str(target) + '.s3part'
        )

        last = None

        for attempt in range(5):
            offset = (
                part.stat().st_size
                if part.exists()
                else 0
            )

            if offset > size:
                part.unlink(missing_ok=True)
                offset = 0

            if offset == size and part.is_file():
                part.replace(target)

                with lock:
                    completed += 1
                    complete_bytes += size

                return

            try:
                options = {
                    'Bucket': bucket,
                    'Key': key
                }

                if offset:
                    options['Range'] = f'bytes={offset}-'

                response = s3.get_object(**options)

                try:
                    if offset and not response.get(
                        'ContentRange', ''
                    ).startswith(f'bytes {offset}-'):
                        raise IOError(
                            f'S3 ignored/mismatched Range for {key}'
                        )

                    mode = 'ab' if offset else 'wb'

                    with open(part, mode) as output:
                        for chunk in response['Body'].iter_chunks(
                            chunk_size=2**20
                        ):
                            if chunk:
                                output.write(chunk)

                                with lock:
                                    downloaded += len(chunk)

                finally:
                    response['Body'].close()

                if part.stat().st_size != size:
                    raise IOError(
                        f'Incomplete {key}: '
                        f'{part.stat().st_size}/{size}'
                    )

                part.replace(target)

                with lock:
                    completed += 1
                    complete_bytes += size

                return

            except Exception as exc:
                last = exc

                if attempt < 4:
                    time.sleep(
                        min(2**attempt, 16)
                    )

        raise RuntimeError(
            f'[S3 {domain}] Failed: {key}\n{last}'
        )

    # --------------------------------------------------------
    # Execute parallel downloads
    # --------------------------------------------------------

    monitor = Thread(
        target=reporter,
        daemon=True
    )

    monitor.start()

    try:
        with ThreadPoolExecutor(
            max_workers=workers
        ) as pool:

            # Bounded queue prevents excessive memory usage.
            window = workers * 12

            for idx in range(
                0,
                len(missing),
                window
            ):
                futures = [
                    pool.submit(get_one, item)
                    for item in missing[idx:idx + window]
                ]

                for future in as_completed(futures):
                    future.result()

    finally:
        stop.set()
        monitor.join(timeout=3)

    # --------------------------------------------------------
    # Final transfer report
    # --------------------------------------------------------

    elapsed = max(
        time.monotonic() - started,
        1
    )

    print(
        f'[S3 {domain}] COMPLETE: '
        f'{len(objects):,} files | '
        f'{total_bytes / 2**20:,.1f} MiB | '
        f'{elapsed / 60:.1f} minutes',
        flush=True
    )

    return {
        'domain': domain,
        'method': 'physionet_public_s3_boto3',
        'url': spec['s3'],
        'file_count': len(objects),
        'bytes': total_bytes,
        'ptb_100hz_only': bool(
            domain == 'PTBXL'
            and PTB_100HZ_ONLY
        )
    }


# ============================================================
# SELECT S3 OR ZIP BACKEND
# ============================================================

def get_source(item):
    hospital, spec = item

    if DOWNLOAD_BACKEND == 'aws_s3':
        return sync_from_official_s3(
            hospital,
            spec
        )

    if DOWNLOAD_BACKEND != 'zip':
        raise ValueError(
            "DOWNLOAD_BACKEND must be 'aws_s3' or 'zip'"
        )

    archive = download(
        spec['url'],
        RAW / spec['name']
    )

    extract_safe(
        archive,
        RAW / hospital,
        ptb_only=(
            hospital == 'PTBXL'
            and PTB_100HZ_ONLY
        )
    )

    return {
        'domain': hospital,
        'method': 'physionet_zip',
        'url': spec['url'],
        'sha256': digest(archive)
    }


# ============================================================
# DOWNLOAD BOTH DATASETS
# ============================================================

print('[DOWNLOAD CELL] Starting official S3 datasets (PTB-XL + Arrhythmia).', flush=True)
# Install boto3 once to avoid concurrent pip installations.
if DOWNLOAD_BACKEND == 'aws_s3':
    ensure_boto3()

with ThreadPoolExecutor(
    max_workers=min(2, CFG['download_workers'])
) as pool:
    source_log = list(
        pool.map(get_source, SOURCES.items())
    )

write_json(
    OUT / 'download_provenance.json',
    source_log
)

# Paths required by later notebook cells.
PTBROOT = next(
    (RAW / 'PTBXL').rglob('ptbxl_database.csv')
).parent

ARRROOT = next(
    (RAW / 'Arrhythmia').rglob(
        'ConditionNames_SNOMED-CT.csv'
    )
).parent


# ============================================================
# OFFICIAL SHA-256 MANIFESTS
# ============================================================

checksum_file = PTBROOT / 'SHA256SUMS.txt'

if not checksum_file.exists():
    checksum_file = download(
        'https://physionet.org/files/ptb-xl/'
        '1.0.3/SHA256SUMS.txt',
        RAW / 'ptbxl_SHA256SUMS.txt'
    )


def parse_sums(path):
    out = {}

    for line in Path(path).read_text().splitlines():
        parts = line.split(maxsplit=1)

        if (
            len(parts) == 2
            and re.fullmatch(
                r'[0-9a-fA-F]{64}',
                parts[0]
            )
        ):
            rel = (
                parts[1]
                .lstrip('*')
                .removeprefix('./')
            )

            out[rel] = parts[0].lower()

    assert out, (
        f'Invalid official SHA-256 manifest: {path}'
    )

    return out


# ============================================================
# VERIFY PTB-XL METADATA
# ============================================================

PTB_HASH = parse_sums(checksum_file)


def verify_ptb(rel):
    rel = str(rel)

    assert rel in PTB_HASH, (
        f'Missing published SHA256 for {rel}'
    )

    assert digest(PTBROOT / rel) == PTB_HASH[rel], (
        f'Corrupt PTB-XL file: {rel}'
    )


for name in (
    'ptbxl_database.csv',
    'scp_statements.csv'
):
    verify_ptb(name)


# ============================================================
# ARRHYTHMIA AND CHALLENGE MANIFESTS
# ============================================================

arr_sums = ARRROOT / 'SHA256SUMS.txt'

if not arr_sums.exists():
    arr_sums = download(
        'https://physionet.org/files/'
        'ecg-arrhythmia/1.0.0/SHA256SUMS.txt',
        RAW / 'arrhythmia_SHA256SUMS.txt'
    )

challenge_sums = download(
    'https://physionet.org/files/'
    'challenge-2021/1.0.3/SHA256SUMS.txt',
    RAW / 'challenge_SHA256SUMS.txt'
)

ARR_HASH = parse_sums(arr_sums)
CH_HASH = parse_sums(challenge_sums)


# ============================================================
# VERIFY CHINESE HOSPITAL PROVENANCE
# ============================================================

HASH_DOMAIN = {}
ID_ORIGIN = {}

for rel, sha in CH_HASH.items():
    if not rel.endswith('.mat'):
        continue

    domain = (
        'Shaoxing'
        if '/chapman_shaoxing/' in '/' + rel
        else (
            'Ningbo'
            if '/ningbo/' in '/' + rel
            else None
        )
    )

    if domain:
        HASH_DOMAIN.setdefault(
            sha,
            set()
        ).add(domain)

        rid = Path(rel).stem

        assert rid not in ID_ORIGIN, (
            f'Duplicate official record identity: {rid}'
        )

        ID_ORIGIN[rid] = (domain, sha)


# ============================================================
# CONSTRUCT VERIFIED HOSPITAL COHORTS
# ============================================================

HOSPITAL_HEADERS = {
    'Shaoxing': [],
    'Ningbo': []
}

membership = []

for rel, sha in ARR_HASH.items():
    if not rel.endswith('.mat'):
        continue

    rid = Path(rel).stem

    if rid in ID_ORIGIN:
        domain, official_sha = ID_ORIGIN[rid]

        assert sha == official_sha, (
            f'Waveform differs across releases: {rid}'
        )

    else:
        matched = HASH_DOMAIN.get(
            sha,
            set()
        )

        assert len(matched) == 1, (
            f'Cannot verify hospital origin for {rel}; '
            'stopping before training.'
        )

        domain = next(iter(matched))

    hp = (
        ARRROOT / rel
    ).with_suffix('.hea')

    assert (
        hp.exists()
        and (ARRROOT / rel).exists()
    ), (
        f'Incomplete recording: {rel}'
    )

    HOSPITAL_HEADERS[domain].append(hp)

    membership.append({
        'record': hp.stem,
        'domain': domain,
        'waveform_sha256': sha,
        'relative_path': rel
    })


# ============================================================
# VERIFY EXPECTED HOSPITAL COUNTS
# ============================================================

counts = {
    name: len(headers)
    for name, headers in HOSPITAL_HEADERS.items()
}

assert counts == {
    'Shaoxing': 10247,
    'Ningbo': 34905
}, (
    f'Unexpected hospital counts: {counts}'
)

pd.DataFrame(membership).to_csv(
    OUT / 'tables/verified_hospital_membership.csv',
    index=False
)

write_json(
    OUT / 'membership_manifest_hashes.json',
    {
        'arrhythmia': digest(arr_sums),
        'challenge': digest(challenge_sums),
        'ptbxl': digest(checksum_file)
    }
)


# ============================================================
# PRESERVE DATASET LICENSES
# ============================================================

(OUT / 'dataset_licenses').mkdir(
    exist_ok=True
)

for domain, root in [
    ('PTBXL', PTBROOT),
    ('Arrhythmia', ARRROOT)
]:
    license_file = root / 'LICENSE.txt'

    if license_file.exists():
        shutil.copy2(
            license_file,
            OUT / 'dataset_licenses'
            / f'{domain}_LICENSE.txt'
        )


# ============================================================
# FINAL STATUS
# ============================================================

print(
    '[SUCCESS] Official PhysioNet datasets present.',
    flush=True
)

print(
    '[SUCCESS] Checksum manifests processed.',
    flush=True
)

print(
    '[SUCCESS] Verified hospital counts:',
    counts,
    flush=True
)

print(
    '[SUCCESS] Download cell completed.',
    flush=True
)


# Georgia is an independent 12-lead ECG source in the official Challenge 2021 release.
# Challenge 2021 does not advertise credential-free S3 CLI access, so this
# cohort is fetched via its published HTTPS file endpoint, using concurrent
# resumable downloads and the official SHA256SUMS manifest.
GEORGIA_ROOT = RAW / 'Georgia'
GEORGIA_ROOT.mkdir(exist_ok=True)
GEORGIA_PREFIX = 'training/georgia/'
GEORGIA_HASH = {p[len(GEORGIA_PREFIX):]: h for p, h in CH_HASH.items()
                if p.startswith(GEORGIA_PREFIX) and p.endswith(('.hea', '.mat'))}
assert any(p.endswith('.hea') for p in GEORGIA_HASH), 'Challenge checksum manifest does not list Georgia headers'
GEORGIA_BASE = 'https://physionet.org/files/challenge-2021/1.0.3/training/georgia/'

def fetch_georgia(rel):
    from urllib.parse import quote
    import concurrent.futures
    dest = GEORGIA_ROOT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and digest(dest) == GEORGIA_HASH[rel]:
        return dest
    part = Path(str(dest) + '.part')
    url = GEORGIA_BASE + quote(rel, safe='/')
    for attempt in range(5):
        try:
            offset = part.stat().st_size if part.exists() else 0
            headers = {'Range': f'bytes={offset}-'} if offset else {}
            with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as r:
                r.raise_for_status()
                mode = 'ab' if offset and r.status_code == 206 else 'wb'
                with open(part, mode) as fh:
                    for chunk in r.iter_content(chunk_size=2**20):
                        if chunk: fh.write(chunk)
            if digest(part) != GEORGIA_HASH[rel]:
                part.unlink(missing_ok=True)
                raise ValueError(f'Official SHA256 mismatch for Georgia {rel}')
            part.replace(dest)
            return dest
        except (requests.RequestException, OSError, ValueError):
            if attempt == 4: raise
            time.sleep(min(2**attempt, 16))

GEORGIA_HEADER_FILES = sorted(x for x in GEORGIA_HASH if x.endswith('.hea'))
# Download headers first. Only eligible AFIB/SR 10-s signals need waveform transfer.
print(f'[Georgia] Starting {len(GEORGIA_HEADER_FILES):,} checksum-verified header downloads.', flush=True)
with ThreadPoolExecutor(max_workers=16) as pool:
    future_map = {pool.submit(fetch_georgia, rel): rel for rel in GEORGIA_HEADER_FILES}
    for future in tqdm(as_completed(future_map), total=len(future_map),
                       desc='Georgia verified headers', mininterval=2):
        future.result()
source_log.append(dict(domain='Georgia', url=GEORGIA_BASE,
    method='parallel official HTTPS + SHA256SUMS',
    checksum_manifest_sha256=digest(challenge_sums),
    header_count=len(GEORGIA_HEADER_FILES)))
write_json(OUT/'download_provenance.json', source_log)

# ============================================================================
# Notebook cell 7 (zero-based 6): markdown
# ============================================================================
# ## 05–07 · Labels, manifest and exclusions
# AF is `AFIB` / SNOMED `164889003`; SR is `SR` / `426783006`. Flutter, simultaneous AF+SR and other rhythm labels are excluded. Non-rhythm diagnoses may coexist: SR does **not** mean a healthy heart. PTB-XL likelihood 0 means unknown confidence, not absence, so code presence defines labels. All exclusions are recorded. Chinese patient proxies use one-ECG-per-patient documentation in the Challenge paper; they are explicitly marked in metadata.
# Georgia is the third domain instead of Ningbo. Only Challenge Georgia recordings with verified 12-lead 10-second waveforms and eligible AFIB/SR codes are used; its record IDs are not verified patient identities.

# ============================================================================
# Notebook cell 8 (zero-based 7): code
# ============================================================================
# Cell 7 (notebook cell 8): AFIB versus strict sinus rhythm labels.
# PhysioNet/CinC Challenge 2021 Figure 4 reports zero AFIB labels for Ningbo:
# https://moody-challenge.physionet.org/2021/papers/2021ChallengePaperPMEA.pdf
# This code audits the label availability; it does not relabel flutter as AFIB.

LEADS = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
AFIB_CODE, SR_CODE, AFL_CODE = '164889003', '426783006', '164890007'

# Strict SR: do not merge sinus bradycardia/tachycardia/arrhythmia with SR.
OTHER_RHYTHMS = {
    '164890007', '426177001', '427084000', '427393009', '426761007',
    '713422000', '233896004', '233897008', '195101003', '426648003'
}

condition_table = pd.read_csv(ARRROOT / 'ConditionNames_SNOMED-CT.csv')
condition_table.columns = condition_table.columns.str.strip()
condition_codes = dict(zip(condition_table['Acronym Name'],
                           condition_table['Snomed_CT'].astype(str)))
assert (condition_codes['AFIB'] == AFIB_CODE
        and condition_codes['AF'] == AFL_CODE
        and condition_codes['SR'] == SR_CODE), 'Official label dictionary changed.'

ptb = pd.read_csv(PTBROOT / 'ptbxl_database.csv')
scp = pd.read_csv(PTBROOT / 'scp_statements.csv', index_col=0)
ptb_rhythms = set(scp.index[
    pd.to_numeric(scp['rhythm'], errors='coerce').fillna(0).eq(1)
])


def binary_label(labels, af, sr, other):
    codes = set(labels)
    if codes & set(other):
        return None, 'other_or_conflicting_rhythm'
    if (af in codes) == (sr in codes):
        return None, 'neither_or_both_AF_SR'
    return int(af in codes), ''


def read_header_comments(header_path):
    """Read verified WFDB #metadata without parsing optional malformed dates.

    The waveform reader will still use WFDB in the signal-processing cell.
    No global datetime or wfdb library monkeypatch is needed for labels.
    """
    comments = {}
    with open(header_path, 'r', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            if line.startswith('#') and ':' in line:
                key, value = line[1:].split(':', 1)
                comments[key.strip()] = value.strip()
    if not comments.get('Dx'):
        raise ValueError(f'Missing #Dx annotation in {header_path}')
    return comments


rows, exclusions = [], []
raw_rhythm_counts = {
    d: {'AFIB_code': 0, 'SR_code': 0, 'AFL_code': 0}
    for d in ['Shaoxing']
}

for r in ptb.itertuples():
    labels = ast.literal_eval(r.scp_codes)
    y, reason = binary_label(labels, 'AFIB', 'SR', ptb_rhythms - {'AFIB', 'SR'})
    rid = f'PTBXL:{r.ecg_id}'
    if y is None:
        exclusions.append(dict(record_id=rid, domain='PTBXL', reason=reason))
        continue
    rows.append(dict(
        record_id=rid, patient_id=f'PTBXL:{int(r.patient_id)}',
        domain='PTBXL', patient_basis='explicit_patient_id', label=y,
        path=str(PTBROOT / r.filename_lr), fold=int(r.strat_fold),
        labels=json.dumps(labels), age=r.age, sex=str(r.sex)
    ))

# Ningbo is excluded from AFIB-vs-SR: it has no AFIB-labelled records.
# Do not add Ningbo records to rows/manifest or later rotations.
for hospital in ['Shaoxing']:
    for hp in tqdm(sorted(HOSPITAL_HEADERS[hospital]),
                   desc=hospital + ' headers'):
        rel = hp.relative_to(ARRROOT).as_posix()
        assert digest(hp) == ARR_HASH[rel], f'Header SHA256 mismatch: {rel}'
        comments = read_header_comments(hp)
        labels = {x.strip() for x in comments['Dx'].split(',') if x.strip()}

        for code, key in [(AFIB_CODE, 'AFIB_code'),
                          (SR_CODE, 'SR_code'),
                          (AFL_CODE, 'AFL_code')]:
            raw_rhythm_counts[hospital][key] += int(code in labels)

        y, reason = binary_label(labels, AFIB_CODE, SR_CODE, OTHER_RHYTHMS)
        rid = f'{hospital}:{hp.stem}'
        if y is None:
            exclusions.append(dict(record_id=rid, domain=hospital,
                                   reason=reason))
            continue
        rows.append(dict(
            record_id=rid, patient_id=rid, domain=hospital,
            patient_basis='documented_one_record_per_patient_proxy',
            label=y, path=str(hp.with_suffix('')), fold=0,
            labels=json.dumps(sorted(labels)), age=comments.get('Age'),
            sex=comments.get('Sex')
        ))

# Read Georgia header diagnoses, excluding non-10-second recordings to preserve
# the unchanged signal shape (12 leads x 1000 samples at 100 Hz).
GEORGIA_WAVEFORM_REL = []
for rel in tqdm(GEORGIA_HEADER_FILES, desc='Georgia label preflight'):
    hp = GEORGIA_ROOT / rel
    comments = read_header_comments(hp)
    labels = {c.strip() for c in comments['Dx'].split(',') if c.strip()}
    rid = f'Georgia:{hp.stem}'
    y, reason = binary_label(labels, AFIB_CODE, SR_CODE, OTHER_RHYTHMS)
    if y is None:
        exclusions.append(dict(record_id=rid, domain='Georgia', reason=reason))
        continue
    fields = hp.read_text(encoding='utf-8', errors='replace').splitlines()[0].split()
    if len(fields) < 4 or int(fields[1]) != 12 or abs(int(fields[3]) / float(fields[2]) - 10) > 1e-6:
        exclusions.append(dict(record_id=rid, domain='Georgia', reason='not_12_leads_10_seconds'))
        continue
    mat_rel = str(Path(rel).with_suffix('.mat'))
    assert mat_rel in GEORGIA_HASH, f'Missing published Georgia waveform checksum: {mat_rel}'
    GEORGIA_WAVEFORM_REL.append(mat_rel)
    rows.append(dict(record_id=rid, patient_id=rid, domain='Georgia',
        patient_basis='record_id_proxy_not_verified_unique_patient',
        label=y, path=str(hp.with_suffix('')), fold=0,
        labels=json.dumps(sorted(labels)), age=comments.get('Age'),
        sex=comments.get('Sex')))
# A preflight guard prevents expensive waveform transfers when label eligibility fails.
preflight = pd.DataFrame(rows)
pre_counts = preflight.groupby(['domain', 'label']).size().unstack(fill_value=0)
assert pre_counts.reindex(index=['PTBXL','Shaoxing','Georgia'], columns=[0,1], fill_value=0).min().min() >= 10, \
    f'Third cohort has insufficient AFIB/SR labels: {pre_counts}'
with ThreadPoolExecutor(max_workers=16) as pool:
    list(tqdm(pool.map(fetch_georgia, GEORGIA_WAVEFORM_REL),
              total=len(GEORGIA_WAVEFORM_REL), desc='Georgia verified eligible waveforms'))

manifest = pd.DataFrame(rows)
EXPECTED_DOMAINS = {'PTBXL', 'Shaoxing', 'Georgia'}
actual_domains = set(manifest['domain'].unique())
assert actual_domains == EXPECTED_DOMAINS, (
    f'Unexpected AFIB/SR domains: {actual_domains}; expected {EXPECTED_DOMAINS}. '
    'Ningbo is excluded because it has no AFIB cases. Restart the runtime '
    'and run this entire updated script from the top; do not mix old cells.')
assert set(CFG['targets']).issubset(EXPECTED_DOMAINS), (
    f'Invalid targets {CFG["targets"]}; use Georgia (pilot) or only the three '
    'verified AFIB/SR domains (full).')
assert not manifest.empty, 'No AFIB/SR records were selected.'
assert not manifest.record_id.duplicated().any(), 'Duplicate record identifiers.'

# Save the raw-code audit and the exact included classes before validation.
raw_counts = pd.DataFrame.from_dict(raw_rhythm_counts, orient='index')
raw_counts.index.name = 'domain'
raw_counts.to_csv(OUT / 'tables' / 'hospital_raw_rhythm_code_counts.csv')
counts = (manifest.groupby(['domain', 'label']).size().unstack(fill_value=0)
          .reindex(index=['PTBXL', 'Shaoxing', 'Georgia'],
                   columns=[0, 1], fill_value=0))
counts.columns = ['strict_SR', 'AFIB']
counts.to_csv(OUT / 'tables' / 'afib_strict_sr_domain_counts.csv')
pd.DataFrame(exclusions, columns=['record_id', 'domain', 'reason']).to_csv(
    OUT / 'tables' / 'label_exclusions.csv', index=False)
pd.DataFrame([
    ['PTBXL', 'AFIB', 1], ['PTBXL', 'SR', 0],
    ['Chinese', AFIB_CODE, 1], ['Chinese', SR_CODE, 0]
], columns=['scheme', 'code', 'label']).to_csv(
    OUT / 'tables' / 'label_mapping.csv', index=False)

print('\nRaw Chinese-hospital diagnosis-code counts:')
print(raw_counts.to_string())
print('\nEligible AFIB versus strict-SR class counts:')
print(counts.to_string())

assert (counts[['strict_SR','AFIB']] >= 10).all().all(), \
    'Eligible Georgia/Shaoxing/PTBXL counts are insufficient for stratified training.'
print('Verified three-domain AFIB-versus-strict-SR label manifest.')

# ============================================================================
# Notebook cell 9 (zero-based 8): markdown
# ============================================================================
# ## 08–12 · Physical units, anti-aliasing, audit and disk-backed cache
# Standardize the 12 lead order, convert physical units to mV, require exactly 10 seconds, and polyphase resample to 100 Hz. Save float32 disk-backed arrays; never accumulate all 500 Hz recordings in RAM. The alternative uses a 0.5–40 Hz bandpass. No amplitude clipping or per-record variance normalization is applied. Train-only channel means and standard deviations are computed later.
#
# Hash raw physical waveforms and processed waveforms. Remove all members of exact duplicate groups across the entire cohort before splitting (a predeclared integrity audit, not target-based tuning). This cannot identify near-duplicates or re-identify patients. Audit failures above 2% stop execution instead of silently discarding a broken cohort.

# ============================================================================
# Notebook cell 10 (zero-based 9): code
# ============================================================================
def physical_ecg(path):
    rec=wfdb.rdrecord(str(path),physical=True)
    names=[s.upper() for s in rec.sig_name]
    assert len(names)==12 and set(names)==set(LEADS), 'missing_or_duplicate_leads'
    assert abs(rec.sig_len/rec.fs-10)<1e-6, 'wrong_duration'
    order=[names.index(s) for s in LEADS]
    x=np.asarray(rec.p_signal[:,order],dtype=np.float64).T
    factors={'mv':1.,'uv':.001,'µv':.001,'μv':.001,'v':1000.}
    for j,k in enumerate(order):
        unit=rec.units[k].lower()
        assert unit in factors, f'unknown_unit:{unit}'
        x[j]*=factors[unit]
    assert np.isfinite(x).all(), 'nonfinite'
    assert np.all(np.ptp(x,axis=1)>1e-5), 'flat_lead'
    assert np.max(np.abs(x))<100, 'implausible_amplitude_check_units'
    return x, float(rec.fs)
def process_signal(x,fs,filtered=False):
    from fractions import Fraction
    ratio=Fraction(100/float(fs)).limit_denominator(10000)
    x=resample_poly(x,ratio.numerator,ratio.denominator,axis=-1) if fs!=100 else x.copy()
    assert x.shape==(12,1000) and np.isfinite(x).all(), 'bad_processed_shape'
    if filtered: x=sosfiltfilt(butter(3,[.5,40],btype='bandpass',fs=100,output='sos'),x,axis=-1)
    return np.ascontiguousarray(x,dtype=np.float32)
# Cache files are rebuilt on each run; no stale-cache reuse across protocol changes.
CACHE_RUN=CACHE/RUN; CACHE_RUN.mkdir(parents=True,exist_ok=True)
X=np.lib.format.open_memmap(CACHE_RUN/'minimal.npy',mode='w+',dtype='float32',shape=(len(manifest),12,1000))
XF=np.lib.format.open_memmap(CACHE_RUN/'filtered.npy',mode='w+',dtype='float32',shape=X.shape)
valid=[]; signal_failures=[]
for idx,r in tqdm(enumerate(manifest.itertuples()),total=len(manifest),desc='Physical ECG and cache'):
    try:
        if r.domain=='PTBXL':
            rel=Path(r.path).relative_to(PTBROOT)
            verify_ptb(str(rel)+'.hea'); verify_ptb(str(rel)+'.dat')
        elif r.domain=='Georgia':
            raw_mat=Path(r.path+'.mat'); rel=raw_mat.relative_to(GEORGIA_ROOT).as_posix()
            assert digest(raw_mat)==GEORGIA_HASH[rel], f'Georgia waveform SHA256 mismatch: {rel}'
        else:
            raw_mat=Path(r.path+'.mat'); rel=raw_mat.relative_to(ARRROOT).as_posix()
            assert digest(raw_mat)==ARR_HASH[rel], f'Waveform SHA256 mismatch: {rel}'
        x,fs=physical_ecg(r.path)
        raw_hash=hashlib.sha256(np.round(x,6).astype('<f4').tobytes()).hexdigest()
        a=process_signal(x,fs); b=process_signal(x,fs,True)
        X[idx]=a; XF[idx]=b
        valid.append(dict(cache_index=idx,fs_original=fs,leads='|'.join(LEADS),
            units='mV',raw_hash=raw_hash,signal_hash=hashlib.sha256(a.astype('<f4').tobytes()).hexdigest(),
            rms=float(np.sqrt(np.mean(a*a))),hf_ratio=float(np.mean(np.diff(a,axis=-1)**2)/(np.mean(a*a)+1e-12))))
    except (AssertionError,ValueError,OSError) as e:
        signal_failures.append(dict(record_id=r.record_id,domain=r.domain,reason=str(e)))
X.flush(); XF.flush()
pd.DataFrame(exclusions+signal_failures).to_csv(OUT/'tables/exclusions.csv',index=False)
assert len(signal_failures)/len(manifest)<.02, 'More than 2% signal failures; inspect exclusions before continuing.'
# Compare the eligible classes at each integrity-audit stage. These tables are
# recorded even if a cohort becomes unusable, so no diagnosis is fabricated.
def audit_class_counts(frame, filename):
    matrix = (frame.groupby(['domain', 'label']).size().unstack(fill_value=0)
              .reindex(index=['PTBXL', 'Shaoxing', 'Georgia'],
                       columns=[0, 1], fill_value=0))
    matrix.columns = ['strict_SR', 'AFIB']
    matrix.to_csv(OUT / 'tables' / filename)
    print(f"\n{filename}:\n{matrix.to_string()}", flush=True)
    return matrix

audit_class_counts(manifest, 'class_counts_before_signal_audit.csv')
quality = pd.DataFrame(valid).set_index('cache_index')
manifest = manifest.join(quality, how='inner')
manifest['cache_index'] = manifest.index
audit_class_counts(manifest, 'class_counts_after_signal_audit.csv')

# An exact-duplicate family may contain multiple identical recordings with the
# same label and domain. The previous code used duplicated(keep=False) and
# removed *every* member. Retain one deterministic representative when all
# members agree on domain/label; discard the entire family if there is a label
# conflict or cross-domain overlap (to prevent train/test contamination).
# Union both raw and processed hashes: duplicates connected through either
# representation belong to one family, including transitive matches.
parent = list(range(len(manifest)))
def root_of(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def unite(a, b):
    ra, rb = root_of(a), root_of(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

for hash_column in ('raw_hash', 'signal_hash'):
    first_seen = {}
    for pos, sha in enumerate(manifest[hash_column]):
        if sha in first_seen:
            unite(pos, first_seen[sha])
        else:
            first_seen[sha] = pos

families = {}
for pos in range(len(manifest)):
    families.setdefault(root_of(pos), []).append(pos)

remove_positions = set()
duplicate_audit = []
for positions in families.values():
    if len(positions) == 1:
        continue
    family = manifest.iloc[positions]
    consistent = (family.domain.nunique() == 1 and family.label.nunique() == 1)
    # Within-domain records with a verified patient ID must agree as well;
    # for proxy patient IDs, one record per exact waveform is retained.
    if consistent and family.patient_basis.eq('explicit_patient_id').all():
        consistent = family.patient_id.nunique() == 1
    if consistent:
        # Stable selection independent of dataframe ordering.
        keep_pos = min(positions, key=lambda i: str(manifest.iloc[i].record_id))
        reason = 'redundant_identical_record'
    else:
        keep_pos = None
        reason = 'conflicting_or_cross_domain_duplicate'
    for pos in positions:
        if pos != keep_pos:
            remove_positions.add(pos)
        duplicate_audit.append(dict(
            record_id=manifest.iloc[pos].record_id,
            domain=manifest.iloc[pos].domain,
            label=int(manifest.iloc[pos].label),
            action='retained' if pos == keep_pos else 'excluded',
            reason=reason,
            family_size=len(positions)))

pd.DataFrame(duplicate_audit, columns=[
    'record_id', 'domain', 'label', 'action', 'reason', 'family_size'
]).to_csv(OUT / 'tables' / 'duplicate_family_audit.csv', index=False)
isdup = np.array([i in remove_positions for i in range(len(manifest))], dtype=bool)
manifest.iloc[np.flatnonzero(isdup)].to_csv(
    OUT / 'tables' / 'duplicate_exclusions.csv', index=False)
manifest = manifest.iloc[np.flatnonzero(~isdup)].reset_index(drop=True)
manifest.to_csv(OUT / 'tables' / 'dataset_manifest.csv', index=False)
post_counts = audit_class_counts(manifest, 'class_counts_after_duplicate_audit.csv')
write_json(OUT / 'data_audit.json', dict(
    eligible_before_signal_audit=len(rows),
    signal_failures=len(signal_failures),
    duplicates_removed=int(isdup.sum()),
    duplicate_families=sum(len(g) > 1 for g in families.values()),
    retained=len(manifest), patient_proxy_limitation=True))
# Stratified splits and the pilot/full external evaluations need enough cases
# in every class. Do not hide a genuine missing AFIB class by skipping a domain.
missing = [(d, cls, int(post_counts.loc[d, cls]))
           for d in post_counts.index for cls in ['strict_SR', 'AFIB']
           if post_counts.loc[d, cls] < 40]
if missing:
    failure_summary = (pd.DataFrame(signal_failures)
                       .groupby(['domain', 'reason']).size().reset_index(name='n')
                       if signal_failures else pd.DataFrame(columns=['domain','reason','n']))
    failure_summary.to_csv(OUT / 'tables' / 'signal_failure_summary.csv', index=False)
    raise RuntimeError(
        'Not enough verified AFIB/SR recordings after signal validation and '
        f'deduplication: {missing}. At least 40/class/domain are required '
        'for the planned 4-way stratified splits and calibration subdivision. '
        'Inspect class_counts_before_signal_audit.csv, '
        'class_counts_after_signal_audit.csv, '
        'class_counts_after_duplicate_audit.csv, '
        'duplicate_family_audit.csv, and signal_failure_summary.csv under '
        f'{OUT / "tables"}. Do not relabel atrial flutter or suppress the audit.')
# Chunked finite checks avoid large temporary RAM allocations.
for a in [X,XF]:
    for ids in np.array_split(manifest.cache_index.to_numpy(),max(1,len(manifest)//256)):
        assert np.isfinite(a[ids]).all()
manifest.groupby(['domain','label']).size().unstack().plot.bar()
plt.ylabel('Eligible recordings'); plt.tight_layout(); plt.savefig(OUT/'figures/cohort_counts.png',dpi=160); plt.show(); plt.close()

# ============================================================================
# Notebook cell 11 (zero-based 10): markdown
# ============================================================================
# ### Patient partitions and sealed external domains
# PTB-XL folds 1–7 train, 8 validation, 9 calibration, 10 source test. Non-PTB-XL sources use stratified 70/10/10/10 patient partitions (record-ID proxies where verified patient identifiers are unavailable). Calibration is subdivided by patient: one half fits temperature, the other chooses the operating threshold (Youden J). Holdout labels are used only in predeclared eligibility/integrity audits until the evaluation phase. Pilot subsampling affects training patients only; it does not balance or subsample evaluation data.

# ============================================================================
# Notebook cell 12 (zero-based 11): code
# ============================================================================
def split_domain(frame):
    frame=frame.copy()
    if frame.domain.iloc[0]=='PTBXL':
        frame['split']=np.select([frame.fold<=7,frame.fold==8,frame.fold==9],['train','val','cal'],default='source_test')
    else:
        patients=frame.groupby('patient_id').label.max()
        train,rest=train_test_split(patients.index,test_size=.3,random_state=42,stratify=patients)
        val,rest=train_test_split(rest,test_size=2/3,random_state=42,stratify=patients.loc[rest])
        cal,test=train_test_split(rest,test_size=.5,random_state=42,stratify=patients.loc[rest])
        maps={p:s for s,ids in [('train',train),('val',val),('cal',cal),('source_test',test)] for p in ids}
        frame['split']=frame.patient_id.map(maps)
    patients=frame[frame.split=='cal'].groupby('patient_id').label.max()
    tc,th=train_test_split(patients.index,test_size=.5,random_state=42,stratify=patients)
    frame.loc[frame.patient_id.isin(tc),'split']='cal_temp'
    frame.loc[frame.patient_id.isin(th),'split']='cal_threshold'
    assert frame.groupby('patient_id').split.nunique().max()==1
    return frame
assert set(manifest.domain.unique()) == EXPECTED_DOMAINS, (
    'Invalid domain configuration before splitting: '
    + str(manifest.domain.value_counts().to_dict()))
assert set(CFG['targets']).issubset(EXPECTED_DOMAINS), (
    f'Invalid external evaluation target(s): {CFG["targets"]}')
manifest=pd.concat([split_domain(g) for _,g in manifest.groupby('domain')],ignore_index=True)
manifest.to_csv(OUT/'tables/base_patient_splits.csv',index=False)
def rotation(target):
    f=manifest.copy(); f.loc[f.domain==target,'split']='external'
    if CFG['train_cap']:
        for domain,g in f[f.split=='train'].groupby('domain'):
            if len(g)>CFG['train_cap']:
                p=g.groupby('patient_id').label.max()
                keep,_=train_test_split(p.index,train_size=min(CFG['train_cap']/len(g),.99),random_state=42,stratify=p)
                f.loc[g.index[~g.patient_id.isin(keep)],'split']='unused_pilot'
    assert f.groupby('patient_id').split.nunique().max()==1
    assert f.groupby('signal_hash').split.nunique().max()==1
    split_counts = (f[~f.split.isin(['unused_pilot'])]
                    .groupby(['domain','split','label']).size()
                    .unstack(fill_value=0).reindex(columns=[0,1],fill_value=0))
    split_counts.to_csv(OUT / f'tables/rotation_{target}_class_counts.csv')
    failures = split_counts[split_counts.min(axis=1) < 10]
    if not failures.empty:
        raise RuntimeError(
            f'Insufficient AFIB/SR cases in rotation {target}. '
            f'Class counts (SR=0, AFIB=1):\n{failures.to_string()}\n'
            'Inspect the rotation class-count CSV. Do not train or calibrate '
            'a binary classifier on a one-class split.')
    f.to_csv(OUT/f'tables/splits_external_{target}.csv',index=False)
    return f
ROTATIONS={target:rotation(target) for target in CFG['targets']}
print(manifest.groupby(['domain','split','label']).size().unstack(fill_value=0))

# ============================================================================
# Notebook cell 13 (zero-based 12): markdown
# ============================================================================
# ## 13 · Compact deep networks and alignment objectives
# A small residual Conv1D encoder and a six-module InceptionTime-style encoder use batch normalization, residual connections, global pooling and dropout. Weighted domain/class sampling makes each training batch contain both classes from both sources. ERM and alignment methods share this sampler for a fair comparison. Class-aware alignment matches normalized class centroids across source domains; Deep CORAL matches marginal feature covariances as the standard DG comparator. CORAL weight is fixed at .01 and not claimed to be exhaustively tuned.

# ============================================================================
# Notebook cell 14 (zero-based 13): code
# ============================================================================
import torch
from torch import nn
from torch.nn import functional as F
class Residual(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(cin,cout,7,stride,padding=3,bias=False),nn.BatchNorm1d(cout),nn.ReLU(),
            nn.Conv1d(cout,cout,7,padding=3,bias=False),nn.BatchNorm1d(cout))
        self.skip=nn.Identity() if cin==cout and stride==1 else nn.Sequential(nn.Conv1d(cin,cout,1,stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x): return F.relu(self.net(x)+self.skip(x))
class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv1d(12,32,15,2,7,bias=False),nn.BatchNorm1d(32),nn.ReLU(),
            Residual(32,32),Residual(32,64,2),Residual(64,64),Residual(64,128,2),nn.AdaptiveAvgPool1d(1))
        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(128,1))
    def forward(self,x):
        z=self.encoder(x).squeeze(-1)
        return self.head(z).squeeze(-1),z
class InceptionModule(nn.Module):
    def __init__(self,cin):
        super().__init__()
        self.bottle=nn.Conv1d(cin,16,1,bias=False)
        self.branches=nn.ModuleList([nn.Conv1d(16,16,k,padding=k//2,bias=False) for k in [9,19,39]])
        self.pool=nn.Sequential(nn.MaxPool1d(3,1,1),nn.Conv1d(cin,16,1,bias=False))
        self.bn=nn.BatchNorm1d(64)
    def forward(self,x):
        y=self.bottle(x)
        return F.relu(self.bn(torch.cat([b(y) for b in self.branches]+[self.pool(x)],1)))
class InceptionTime(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.AvgPool1d(2)
        self.blocks=nn.ModuleList([InceptionModule(12 if i==0 else 64) for i in range(6)])
        self.skips=nn.ModuleList([nn.Sequential(nn.Conv1d(12,64,1,bias=False),nn.BatchNorm1d(64)),nn.BatchNorm1d(64)])
        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(64,1))
    def forward(self,x):
        x=self.stem(x); residual=x
        for i,b in enumerate(self.blocks):
            x=b(x)
            if i%3==2:
                x=F.relu(x+self.skips[i//3](residual)); residual=x
        z=x.mean(-1)
        return self.head(z).squeeze(-1),z
def make_model(arch):
    if arch=='resnet': return ResNet()
    if arch=='inception': return InceptionTime()
    raise ValueError(arch)
def alignment_loss(z,y,d,method):
    z=z.float()
    if method=='class':
        z=F.normalize(z,dim=1)
        return sum((z[(d==0)&(y==c)].mean(0)-z[(d==1)&(y==c)].mean(0)).square().sum() for c in [0,1])/2
    if method=='coral':
        a,b=z[d==0],z[d==1]
        a=a-a.mean(0); b=b-b.mean(0)
        return ((a.T@a/(len(a)-1)-b.T@b/(len(b)-1))**2).mean()/4
    return z.sum()*0

# ============================================================================
# Notebook cell 15 (zero-based 14): code
# ============================================================================
(OUT/'bundle/model.py').write_text("import torch\nfrom torch import nn\nfrom torch.nn import functional as F\nclass Residual(nn.Module):\n    def __init__(self,cin,cout,stride=1):\n        super().__init__()\n        self.net=nn.Sequential(nn.Conv1d(cin,cout,7,stride,padding=3,bias=False),nn.BatchNorm1d(cout),nn.ReLU(),\n            nn.Conv1d(cout,cout,7,padding=3,bias=False),nn.BatchNorm1d(cout))\n        self.skip=nn.Identity() if cin==cout and stride==1 else nn.Sequential(nn.Conv1d(cin,cout,1,stride,bias=False),nn.BatchNorm1d(cout))\n    def forward(self,x): return F.relu(self.net(x)+self.skip(x))\nclass ResNet(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.encoder=nn.Sequential(nn.Conv1d(12,32,15,2,7,bias=False),nn.BatchNorm1d(32),nn.ReLU(),\n            Residual(32,32),Residual(32,64,2),Residual(64,64),Residual(64,128,2),nn.AdaptiveAvgPool1d(1))\n        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(128,1))\n    def forward(self,x):\n        z=self.encoder(x).squeeze(-1)\n        return self.head(z).squeeze(-1),z\nclass InceptionModule(nn.Module):\n    def __init__(self,cin):\n        super().__init__()\n        self.bottle=nn.Conv1d(cin,16,1,bias=False)\n        self.branches=nn.ModuleList([nn.Conv1d(16,16,k,padding=k//2,bias=False) for k in [9,19,39]])\n        self.pool=nn.Sequential(nn.MaxPool1d(3,1,1),nn.Conv1d(cin,16,1,bias=False))\n        self.bn=nn.BatchNorm1d(64)\n    def forward(self,x):\n        y=self.bottle(x)\n        return F.relu(self.bn(torch.cat([b(y) for b in self.branches]+[self.pool(x)],1)))\nclass InceptionTime(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.stem=nn.AvgPool1d(2)\n        self.blocks=nn.ModuleList([InceptionModule(12 if i==0 else 64) for i in range(6)])\n        self.skips=nn.ModuleList([nn.Sequential(nn.Conv1d(12,64,1,bias=False),nn.BatchNorm1d(64)),nn.BatchNorm1d(64)])\n        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(64,1))\n    def forward(self,x):\n        x=self.stem(x); residual=x\n        for i,b in enumerate(self.blocks):\n            x=b(x)\n            if i%3==2:\n                x=F.relu(x+self.skips[i//3](residual)); residual=x\n        z=x.mean(-1)\n        return self.head(z).squeeze(-1),z\ndef make_model(arch):\n    if arch=='resnet': return ResNet()\n    if arch=='inception': return InceptionTime()\n    raise ValueError(arch)\ndef alignment_loss(z,y,d,method):\n    z=z.float()\n    if method=='class':\n        z=F.normalize(z,dim=1)\n        return sum((z[(d==0)&(y==c)].mean(0)-z[(d==1)&(y==c)].mean(0)).square().sum() for c in [0,1])/2\n    if method=='coral':\n        a,b=z[d==0],z[d==1]\n        a=a-a.mean(0); b=b-b.mean(0)\n        return ((a.T@a/(len(a)-1)-b.T@b/(len(b)-1))**2).mean()/4\n    return z.sum()*0\n")

# ============================================================================
# Notebook cell 16 (zero-based 15): markdown
# ============================================================================
# ### Train-only normalization and balanced data loading
# Disk-backed data, pinned memory, bounded CPU workers and FP16 autocast reduce memory use. A real forward/backward/optimizer probe chooses a safe batch size with VRAM headroom. OOM recovery restarts a fit at a smaller batch, recording the change. It does not claim to eliminate every possible hardware or memory failure.

# ============================================================================
# Notebook cell 17 (zero-based 16): code
# ============================================================================
def fit_scaler(frame,filtered=False):
    a=XF if filtered else X
    ids=frame.loc[frame.split=='train','cache_index'].to_numpy()
    total=np.zeros(12); squares=np.zeros(12); n=0
    for start in range(0,len(ids),128):
        v=np.asarray(a[ids[start:start+128]],dtype=np.float64)
        total+=v.sum(axis=(0,2)); squares+=(v*v).sum(axis=(0,2)); n+=v.shape[0]*1000
    mu=total/n; sd=np.sqrt(np.maximum(squares/n-mu*mu,1e-8))
    return mu.astype('float32'),sd.astype('float32')
class ECGDataset(Dataset):
    def __init__(self,frame,scaler,filtered=False,augment=False):
        self.frame=frame.reset_index(drop=True); self.ids=self.frame.cache_index.to_numpy()
        self.path=str(CACHE_RUN/('filtered.npy' if filtered else 'minimal.npy')); self.array=None
        self.mu,self.sd=scaler; self.augment=augment
        self.y=self.frame.label.to_numpy(dtype='float32')
        self.d=pd.Categorical(self.frame.domain,categories=sorted(self.frame.domain.unique())).codes.astype('int64')
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        if self.array is None: self.array=np.load(self.path,mmap_mode='r')
        x=(np.array(self.array[self.ids[i]],copy=True)-self.mu[:,None])/self.sd[:,None]
        return torch.from_numpy(x),torch.tensor(self.y[i]),torch.tensor(self.d[i])
class BalancedBatches(Sampler):
    def __init__(self,ds,batch,seed):
        self.batch=batch-(batch%4); self.seed=seed; self.epoch=0
        self.groups=[np.flatnonzero((ds.d==d)&(ds.y==c)) for d in [0,1] for c in [0,1]]
        assert all(len(g)>=2 for g in self.groups), 'Every source needs AF and SR.'
        self.steps=max(1,math.ceil(len(ds)/self.batch))
    def __len__(self): return self.steps
    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch); self.epoch+=1
        for _ in range(self.steps):
            ids=np.concatenate([rng.choice(g,self.batch//4,replace=len(g)<self.batch//4) for g in self.groups])
            rng.shuffle(ids); yield ids.tolist()
def loader(ds,batch,train=False,seed=42):
    opts=dict(num_workers=CFG['workers'],pin_memory=DEVICE.type=='cuda')
    if CFG['workers']: opts.update(prefetch_factor=2,persistent_workers=False)
    if train: return DataLoader(ds,batch_sampler=BalancedBatches(ds,batch,seed),**opts)
    return DataLoader(ds,batch_size=batch,shuffle=False,**opts)
def autocast(): return torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda')
@torch.inference_mode()
def predict(model,ds,batch):
    model.eval(); out=[]
    for x,_,_ in loader(ds,batch):
        with autocast(): logits,_=model(x.to(DEVICE,non_blocking=True))
        out.append(logits.float().cpu().numpy())
    z=np.concatenate(out)
    assert len(z)==len(ds) and np.isfinite(z).all()
    return z
def domain_auc(frame,logits):
    return float(np.mean([roc_auc_score(g.label,logits[g.index.to_numpy()]) for _,g in frame.reset_index(drop=True).groupby('domain')]))
def probe_batch(arch):
    if DEVICE.type!='cuda': return 32
    chosen=None
    for batch in [b for b in [32,64,128,256] if b<=CFG['max_batch']]:
        model=opt=x=y=d=logits=z=loss=None
        try:
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model=make_model(arch).to(DEVICE); opt=torch.optim.AdamW(model.parameters())
            x=torch.randn(batch,12,1000,device=DEVICE); y=torch.arange(batch,device=DEVICE)%2
            d=(torch.arange(batch,device=DEVICE)//2)%2
            with autocast():
                logits,z=model(x); loss=F.binary_cross_entropy_with_logits(logits,y.float())+.1*alignment_loss(z,y,d,'class')
            loss.backward(); opt.step(); torch.cuda.synchronize()
            peak=torch.cuda.max_memory_allocated(); total=torch.cuda.get_device_properties(0).total_memory
            if peak<total*.75: chosen=batch
            else: break
        except torch.cuda.OutOfMemoryError:
            break
        finally:
            del model,opt,x,y,d,logits,z,loss
            gc.collect(); torch.cuda.empty_cache()
    assert chosen, 'Even batch 32 failed. Restart runtime or reduce architecture.'
    return chosen
BATCH={arch:probe_batch(arch) for arch in ['resnet','inception']}
write_json(OUT/'batch_probe.json',BATCH)
print('Measured safe training batches:',BATCH)

# ============================================================================
# Notebook cell 18 (zero-based 17): markdown
# ============================================================================
# ## 14 · Mandatory tiny-subset learning check
# Fit 32 balanced real source examples with no augmentation. Stop if loss fails to fall materially. The test model is discarded and never evaluated as a research result.

# ============================================================================
# Notebook cell 19 (zero-based 18): code
# ============================================================================
first_frame=ROTATIONS[CFG['targets'][0]]
small=first_frame[first_frame.split=='train'].groupby(['domain','label'],group_keys=False).sample(n=8,random_state=42)
scale=fit_scaler(first_frame)
tiny_ds=ECGDataset(small,scale)
tx=torch.stack([tiny_ds[i][0] for i in range(len(tiny_ds))]).to(DEVICE)
ty=torch.tensor(tiny_ds.y,device=DEVICE)
seed_all(42); tiny=make_model('resnet').to(DEVICE); opt=torch.optim.AdamW(tiny.parameters(),lr=.003)
losses=[]
for step in range(180):
    tiny.train(); opt.zero_grad(set_to_none=True)
    logits,_=tiny(tx); loss=F.binary_cross_entropy_with_logits(logits,ty)
    assert torch.isfinite(loss)
    loss.backward(); nn.utils.clip_grad_norm_(tiny.parameters(),1.); opt.step(); losses.append(float(loss.detach()))
    if step>=30 and np.mean(losses[-10:])<.08: break
assert np.mean(losses[-10:])<.5*losses[0], 'Tiny overfit check failed: inspect inputs and labels.'
write_json(OUT/'tiny_overfit_check.json',dict(initial=losses[0],last_mean=float(np.mean(losses[-10:])),steps=len(losses)))
del tiny,opt,tx,ty; gc.collect()
if DEVICE.type=='cuda': torch.cuda.empty_cache()

# ============================================================================
# Notebook cell 20 (zero-based 19): markdown
# ============================================================================
# ## 15–21 · Baselines, ablations, independent seeds and source-only calibration
# Every fit uses the same source validation criterion (mean per-domain AUROC), AdamW, cosine learning-rate decay, gradient clipping, early stopping and best-checkpoint restoration. Time-capped fits are explicitly marked. Baselines and the filtered preprocessing ablation use seed 42; additional independent seeds repeat the validation-selected ResNet alignment weight, including zero if it wins. The exported demo uses this selected ResNet ensemble. InceptionTime and CORAL remain comparators and are reported even when they perform better.
#
# Model selection and calibration use source domains only. This revision is exploratory after review of a prior Georgia run; a repeat Georgia result is not independent confirmation. All rotations are trained and frozen **before any external metrics are printed**. Full-mode researchers should predeclare the protocol before running; seeing results from one rotation must not guide the others. The overall pilot budget can skip later jobs, which are listed as skipped rather than reported as completed experiments.

# ============================================================================
# Notebook cell 21 (zero-based 20): code
# ============================================================================
def atomic_save(obj,path):
    tmp=Path(str(path)+'.tmp'); torch.save(obj,tmp); tmp.replace(path)
def fit_once(frame,arch,method,weight,seed,filtered,batch,tag):
    seed_all(seed); start=time.monotonic(); scaler=fit_scaler(frame,filtered)
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    tr=ECGDataset(frame[frame.split=='train'],scaler,filtered)
    va=ECGDataset(frame[frame.split=='val'],scaler,filtered)
    model=make_model(arch).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG['epochs'])
    amp=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    best=-np.inf; bad=0; history=[]; path=OUT/'models'/f'{tag}.pt'; stop='max_epochs'
    train_loader=loader(tr,batch,True,seed)
    for epoch in range(CFG['epochs']):
        model.train(); loss_sum=0.; align_sum=0.; count=0
        for x,y,d in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); d=d.to(DEVICE,non_blocking=True)
            # Weak amplitude/noise augmentation preserves morphology and lead relationships.
            x=x*(.9+.2*torch.rand(len(x),1,1,device=DEVICE))+.01*torch.randn_like(x)
            opt.zero_grad(set_to_none=True)
            with autocast():
                logits,z=model(x); al=alignment_loss(z,y,d,method)
                loss=F.binary_cross_entropy_with_logits(logits,y)+weight*al
            assert torch.isfinite(loss), 'Nonfinite training loss'
            amp.scale(loss).backward(); amp.unscale_(opt)
            grad=nn.utils.clip_grad_norm_(model.parameters(),1.)
            assert torch.isfinite(grad), 'Nonfinite gradients'
            amp.step(opt); amp.update()
            loss_sum+=float(loss.detach())*len(x); align_sum+=float(al.detach())*len(x); count+=len(x)
            if CFG['fit_seconds'] and time.monotonic()-start>=CFG['fit_seconds']: break
        val_logits=predict(model,va,batch); score=domain_auc(va.frame,val_logits)
        history.append(dict(epoch=epoch+1,loss=loss_sum/count,alignment=align_sum/count,val_auc=score,
                            elapsed=time.monotonic()-start,examples=count))
        if score>best+1e-4:
            best=score; bad=0
            atomic_save(dict(state={k:v.detach().cpu() for k,v in model.state_dict().items()},
                arch=arch,method=method,weight=weight,seed=seed,filtered=filtered,
                mean=torch.from_numpy(scaler[0]),std=torch.from_numpy(scaler[1]),batch=batch,
                val_auc=best,epoch=epoch+1),path)
        else: bad+=1
        sched.step()
        if CFG['fit_seconds'] and time.monotonic()-start>=CFG['fit_seconds']: stop='time_cap'; break
        if bad>=CFG['patience']: stop='early_stop'; break
    pd.DataFrame(history).to_csv(OUT/f'tables/history_{tag}.csv',index=False)
    result=dict(tag=tag,path=str(path),arch=arch,method=method,weight=weight,seed=seed,
        filtered=filtered,val_auc=best,epochs=len(history),seconds=time.monotonic()-start,batch=batch,stop=stop,
        ram_rss_gib=psutil.Process().memory_info().rss/2**30,
        gpu_peak_gib=torch.cuda.max_memory_allocated()/2**30 if DEVICE.type=='cuda' else 0)
    del model,opt,train_loader,tr,va; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    sync_outputs(); return result
OOM_EVENTS=[]
def fit(frame,arch,method,weight,seed,filtered,tag):
    batch=BATCH[arch]
    while batch>=CFG['min_batch']:
        try: return fit_once(frame,arch,method,weight,seed,filtered,batch,tag)
        except torch.cuda.OutOfMemoryError:
            OOM_EVENTS.append(dict(tag=tag,failed_batch=batch)); batch//=2
            gc.collect(); torch.cuda.empty_cache()
            write_json(OUT/'oom_retries.json',OOM_EVENTS)
    raise RuntimeError(f'Fit {tag} cannot fit memory even at minimum batch.')
def load_model(path):
    ck=torch.load(path,map_location='cpu',weights_only=True)
    model=make_model(ck['arch']); model.load_state_dict(ck['state']); model.to(DEVICE).eval()
    return model,ck
def infer_checkpoint(path,frame):
    model,ck=load_model(path)
    ds=ECGDataset(frame,(ck['mean'].numpy(),ck['std'].numpy()),ck['filtered'])
    logits=predict(model,ds,ck['batch'])
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return logits
JOBS=[]; SELECTED={}; SKIPPED=[]
def budget_left():
    return CFG['total_budget_minutes'] is None or (time.monotonic()-START)/60<CFG['total_budget_minutes']-10
for target,frame in ROTATIONS.items():
    candidates=[]
    specs=[('resnet','erm',0.,False),('inception','erm',0.,False)]
    specs += [('resnet','class',w,False) for w in CFG['alignment'] if w>0]
    specs += [('resnet','coral',.01,False),('resnet','erm',0.,True)]
    for arch,method,w,filtered in specs:
        tag=f'{target}_{arch}_{method}_{w}_filter{int(filtered)}_s42'
        if not budget_left(): SKIPPED.append(tag); continue
        r=fit(frame,arch,method,w,42,filtered,tag); r['target']=target; JOBS.append(r)
        print(tag,round(r['val_auc'],4),r['stop'])
        if arch=='resnet' and method in ['erm','class'] and not filtered: candidates.append(r)
        pd.DataFrame(JOBS).to_csv(OUT/'tables/training_summary.csv',index=False)
    assert candidates, 'Budget exhausted before a baseline completed; rerun with a larger time budget.'
    bases=[r for r in JOBS if r['target']==target and r['method']=='erm' and not r['filtered']]
    assert all(r['val_auc']>.55 for r in bases), 'Baseline did not learn. Inspect source data before external testing.'
    winner=max(candidates,key=lambda r:r['val_auc']); SELECTED[target]=[winner['tag']]
    for seed in CFG['seeds'][1:]:
        tag=f'{target}_selected_s{seed}'
        if not budget_left(): SKIPPED.append(tag); continue
        r=fit(frame,'resnet',winner['method'],winner['weight'],seed,False,tag)
        r['target']=target; JOBS.append(r); SELECTED[target].append(tag)
write_json(OUT/'skipped_jobs.json',SKIPPED)
pd.DataFrame(JOBS).to_csv(OUT/'tables/training_summary.csv',index=False)
assert JOBS
print('Training finished. External evaluation has not yet run.')

# ============================================================================
# Notebook cell 22 (zero-based 21): code
# ============================================================================
def fit_temperature(logits,y):
    def nll(log_t):
        z=logits/np.exp(log_t)
        return float(np.mean(np.logaddexp(0,z)-y*z))
    result=minimize_scalar(nll,bounds=(-3,3),method='bounded')
    assert result.success and np.isfinite(result.fun)
    return float(np.exp(result.x))
def choose_threshold(y,p):
    """Original source-calibration Youden-J threshold; retain for comparison."""
    fpr,tpr,threshold=roc_curve(y,p)
    ok=np.isfinite(threshold)&(threshold>=0)&(threshold<=1)
    ids=np.flatnonzero(ok)
    return float(threshold[ids[np.argmax((tpr-fpr)[ids])]])

def source_sensitivity_threshold(frame, probabilities, target_sensitivity=0.97,
                                 min_specificity=0.95):
    """Select on source *calibration* labels only; never look at external labels.

    Prefer the highest threshold meeting sensitivity and specificity constraints
    separately in EVERY source domain. If infeasible, maximize the worst-source
    sensitivity subject to the specificity floor, and mark the target unmet.
    """
    if not (0 < target_sensitivity <= 1 and 0 <= min_specificity <= 1):
        raise ValueError('Invalid source-calibration operating constraints')
    data = frame[['domain','label']].reset_index(drop=True)
    p = np.asarray(probabilities, dtype=np.float64)
    if len(p) != len(data) or not np.isfinite(p).all():
        raise ValueError('Calibration probability length or finiteness mismatch')
    p = np.clip(p, 0., 1.)
    groups = []
    for domain, group in data.groupby('domain',sort=True):
        idx = group.index.to_numpy()
        y = group.label.to_numpy(dtype=int)
        if set(np.unique(y)) != {0,1}:
            raise RuntimeError(f'Both classes required in source calibration: {domain}')
        groups.append((domain, np.sort(p[idx][y==1]), np.sort(p[idx][y==0])))
    # Evaluate attainable decision boundaries including 'all negative'.
    candidates = np.unique(np.r_[0., p, np.nextafter(1., 2.)])
    possible = []
    for threshold in candidates:
        stats = []
        for _, pos, neg in groups:
            sens = 1. - np.searchsorted(pos,threshold,side='left') / len(pos)
            spec = np.searchsorted(neg,threshold,side='left') / len(neg)
            stats.append((float(sens), float(spec)))
        min_sens = min(a for a,_ in stats)
        min_spec = min(b for _,b in stats)
        if min_spec + 1e-12 >= min_specificity:
            possible.append((float(threshold), min_sens, min_spec))
    assert possible, 'No valid source-calibration threshold'
    feasible = [t for t in possible if t[1]+1e-12 >= target_sensitivity]
    # Largest threshold among feasible points minimizes avoidable false positives.
    if feasible:
        chosen = max(feasible, key=lambda t:t[0])
    else:
        # Cannot promise target sensitivity when data cannot support both floors.
        chosen = max(possible, key=lambda t:(t[1],t[2],t[0]))
        print(f'[CALIBRATION] Desired {target_sensitivity:.0%} sensitivity and '
              f'{min_specificity:.0%} specificity were NOT simultaneously '
              f'attainable in every source domain; using observed '
              f'min sensitivity={chosen[1]:.3f}, min specificity={chosen[2]:.3f}.',
              flush=True)
    if not (0 <= chosen[0] <= 1):
        raise RuntimeError('Source calibration cannot select a valid probability threshold; '
                           'inspect source labels, calibration predictions, and cohort shift.')
    return dict(threshold=chosen[0], desired_sensitivity=float(target_sensitivity),
                required_specificity=float(min_specificity),
                achieved_min_source_sensitivity=chosen[1],
                achieved_min_source_specificity=chosen[2],
                target_met=bool(feasible))

def source_calibration_rows(tag,frame,p,selected,youden):
    rows=[]
    for domain,group in frame.reset_index(drop=True).groupby('domain'):
        ix=group.index.to_numpy(); yy=group.label.to_numpy(); pp=np.asarray(p)[ix]
        for name,threshold in [('source_sensitivity',selected['threshold']),('youden',youden)]:
            pred=pp>=threshold
            rows.append(dict(tag=tag,source_domain=domain,operating_point=name,
                threshold=float(threshold),n=len(yy),afib=int(yy.sum()),
                sensitivity=float(pred[yy==1].mean()),
                specificity=float((~pred[yy==0]).mean()),
                source_sensitivity_target_met=selected['target_met']))
    return rows

def entropy(p):
    p=np.clip(p,1e-7,1-1e-7); return -(p*np.log(p)+(1-p)*np.log(1-p))
CALIBRATION={}; ENSEMBLE={}; SOURCE_CALIBRATION_ROWS=[]
for job in JOBS:
    f=ROTATIONS[job['target']]
    tc=f[f.split=='cal_temp'].reset_index(drop=True); th=f[f.split=='cal_threshold'].reset_index(drop=True)
    temp=fit_temperature(infer_checkpoint(job['path'],tc),tc.label.to_numpy())
    p=expit(infer_checkpoint(job['path'],th)/temp)
    original_threshold=choose_threshold(th.label,p)
    op=source_sensitivity_threshold(th,p,
        CFG['source_target_sensitivity'],CFG['source_min_specificity'])
    SOURCE_CALIBRATION_ROWS.extend(source_calibration_rows(job['tag'],th,p,op,original_threshold))
    threshold=op['threshold'] if CFG['operating_point']=='source_sensitivity' else original_threshold
    CALIBRATION[job['tag']]=dict(temperature=temp,threshold=threshold,
                              threshold_youden=original_threshold,
                              threshold_source_sensitivity=op['threshold'],
                              source_operating_diagnostics=op,
                              review_entropy=float(np.quantile(entropy(p),.9)))
for target,tags in SELECTED.items():
    th=ROTATIONS[target].query("split=='cal_threshold'").reset_index(drop=True)
    members=[j for j in JOBS if j['tag'] in tags]
    p=np.mean([expit(infer_checkpoint(j['path'],th)/CALIBRATION[j['tag']]['temperature']) for j in members],axis=0)
    original_threshold=choose_threshold(th.label,p)
    op=source_sensitivity_threshold(th,p,
        CFG['source_target_sensitivity'],CFG['source_min_specificity'])
    SOURCE_CALIBRATION_ROWS.extend(source_calibration_rows(target+'_selected_ensemble',th,p,op,original_threshold))
    threshold=op['threshold'] if CFG['operating_point']=='source_sensitivity' else original_threshold
    ENSEMBLE[target]=dict(tags=tags,threshold=threshold,
        threshold_youden=original_threshold,
        threshold_source_sensitivity=op['threshold'],source_operating_diagnostics=op,
        review_entropy=float(np.quantile(entropy(p),.9)))
pd.DataFrame(SOURCE_CALIBRATION_ROWS).to_csv(
    OUT/'tables/source_calibration_operating_points.csv',index=False)
print('[CALIBRATION] Operating points fitted using source calibration only:',
      CFG['operating_point'],flush=True)
FREEZE=dict(config=CFG,selected=SELECTED,calibration=CALIBRATION,ensemble=ENSEMBLE,
    checkpoint_sha256={j['tag']:digest(j['path']) for j in JOBS},skipped=SKIPPED,
    manifest_sha256=digest(OUT/'tables/dataset_manifest.csv'))
write_json(OUT/'protocol_freeze.json',FREEZE); FREEZE_HASH=digest(OUT/'protocol_freeze.json')
sync_outputs()
print('Protocol frozen:',FREEZE_HASH)

# ============================================================================
# Notebook cell 23 (zero-based 22): markdown
# ============================================================================
# ## 22–24 · Exploratory external evaluation, uncertainty, confidence intervals and failures
# Only now evaluate the external domain. This is an exploratory repeat on the previously examined Georgia cohort. Compare source-calibrated sensitivity-focused and Youden thresholds; neither is tuned using Georgia labels. Report AUROC, AUPRC, sensitivity, specificity, F1, accuracy, Brier score and equal-width 15-bin ECE, with per-record predictions. Patient-cluster bootstrap intervals preserve repeated ECG dependence; paired bootstrap differences compare the selected ensemble with the ERM ResNet using identical resampled patients. Intervals are conditional on fitted models, not a substitute for independent seed variation.
#
# Risk–coverage curves rank predictive entropy (ensemble disagreement is also saved). The review flag uses a source-only entropy cutoff fixed to the 90th percentile of threshold-calibration entropy; external review rates are not forced to 10%. Calibration may still fail under shift. Noise stress is a predeclared synthetic sensitivity analysis, not validation on real home devices.

# ============================================================================
# Notebook cell 24 (zero-based 23): code
# ============================================================================
def metrics(y,p,threshold=.5,weights=None):
    y=np.asarray(y); p=np.clip(np.asarray(p),1e-7,1-1e-7); pred=p>=threshold
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1],sample_weight=weights).ravel()
    w=np.ones(len(y)) if weights is None else np.asarray(weights)
    ece=0.
    for lo,hi in zip(np.linspace(0,1,16)[:-1],np.linspace(0,1,16)[1:]):
        mask=(p>=lo)&(p<hi)
        if w[mask].sum():
            ece+=w[mask].sum()/w.sum()*abs(np.average(y[mask],weights=w[mask])-np.average(p[mask],weights=w[mask]))
    return dict(auroc=roc_auc_score(y,p,sample_weight=w),auprc=average_precision_score(y,p,sample_weight=w),
        sensitivity=tp/max(tp+fn,1),specificity=tn/max(tn+fp,1),f1=f1_score(y,pred,sample_weight=w,zero_division=0),
        accuracy=accuracy_score(y,pred,sample_weight=w),brier=brier_score_loss(y,p,sample_weight=w),ece=ece)
def bootstrap(frame,p,threshold,other=None,other_threshold=None):
    group=pd.factorize(frame.patient_id)[0]; n=group.max()+1; rng=np.random.default_rng(3407); records=[]
    y=frame.label.to_numpy()
    for _ in range(CFG['bootstrap']):
        weights=np.bincount(rng.integers(n,size=n),minlength=n)[group]
        if any(weights[y==c].sum()==0 for c in [0,1]): continue
        a=metrics(y,p,threshold,weights)
        if other is not None:
            b=metrics(y,other,other_threshold,weights); a={k:a[k]-b[k] for k in a}
        records.append(a)
    assert len(records)>=CFG['bootstrap']*.8, 'Too many single-class bootstrap replicates.'
    q=pd.DataFrame(records).quantile([.025,.975]).T
    q.columns=['ci_low','ci_high']; q.index.name='metric'; return q.reset_index()
def risk_curve(y,p,threshold):
    order=np.argsort(entropy(p),kind='stable'); errors=((p>=threshold)!=np.asarray(y))[order]
    return pd.DataFrame(dict(coverage=np.arange(1,len(y)+1)/len(y),risk=np.cumsum(errors)/np.arange(1,len(y)+1)))
PRED={}; RESULTS=[]; CIS=[]; PAIRED=[]; OPERATING_RESULTS=[]
assert digest(OUT/'protocol_freeze.json')==FREEZE_HASH
for j in JOBS:
    assert digest(j['path'])==FREEZE['checkpoint_sha256'][j['tag']], 'Checkpoint modified after freeze'
    for split in ['source_test','external']:
        f=ROTATIONS[j['target']].query('split==@split').reset_index(drop=True)
        z=infer_checkpoint(j['path'],f); c=CALIBRATION[j['tag']]; p=expit(z/c['temperature'])
        df=f[['record_id','patient_id','domain','label','rms','hf_ratio','age','sex']].copy()
        df['logit']=z; df['p_uncalibrated']=expit(z); df['p_af']=p; df['entropy']=entropy(p)
        df['prediction']=(p>=c['threshold']).astype(int); df['review']=df.entropy>c['review_entropy']
        df['threshold']=c['threshold']; df['threshold_youden']=c['threshold_youden']
        df['prediction_youden']=(p>=c['threshold_youden']).astype(int)
        df.to_csv(OUT/f"predictions/{j['tag']}_{split}.csv",index=False)
        PRED[(j['tag'],split)]=df
        for domain,g in df.groupby('domain'):
            RESULTS.append(dict(tag=j['tag'],target=j['target'],split=split,domain=domain,n=len(g),
                **metrics(g.label,g.p_af,c['threshold'])))
            for mode,cut in [('source_sensitivity',c['threshold_source_sensitivity']),
                             ('youden',c['threshold_youden'])]:
                OPERATING_RESULTS.append(dict(tag=j['tag'],target=j['target'],
                    split=split,domain=domain,operating_point=mode,n=len(g),
                    threshold=cut,**metrics(g.label,g.p_af,cut)))
for target,e in ENSEMBLE.items():
    tag=f'{target}_selected_ensemble'
    for split in ['source_test','external']:
        dfs=[PRED[(t,split)] for t in e['tags']]
        assert all(d.record_id.equals(dfs[0].record_id) for d in dfs)
        pp=np.stack([d.p_af.to_numpy() for d in dfs]); p=pp.mean(0)
        df=dfs[0].drop(columns=['logit','p_uncalibrated']).copy()
        df['p_af']=p; df['entropy']=entropy(p); df['variance']=pp.var(0)
        df['mutual_information']=entropy(p)-np.mean(entropy(pp),axis=0)
        df['threshold']=e['threshold']; df['prediction']=(p>=e['threshold']).astype(int)
        df['threshold_youden']=e['threshold_youden']
        df['prediction_youden']=(p>=e['threshold_youden']).astype(int)
        df['review']=df.entropy>e['review_entropy']
        df.to_csv(OUT/f'predictions/{tag}_{split}.csv',index=False); PRED[(tag,split)]=df
        for domain,g in df.groupby('domain'):
            RESULTS.append(dict(tag=tag,target=target,split=split,domain=domain,n=len(g),**metrics(g.label,g.p_af,e['threshold'])))
            for mode,cut in [('source_sensitivity',e['threshold_source_sensitivity']),
                             ('youden',e['threshold_youden'])]:
                OPERATING_RESULTS.append(dict(tag=tag,target=target,
                    split=split,domain=domain,operating_point=mode,n=len(g),
                    threshold=cut,**metrics(g.label,g.p_af,cut)))
            ci=bootstrap(g,g.p_af.to_numpy(),e['threshold']); ci['tag']=tag; ci['domain']=domain; ci['split']=split; CIS.append(ci)
        rc=risk_curve(df.label,df.p_af.to_numpy(),e['threshold'])
        rc.to_csv(OUT/f'tables/risk_coverage_{tag}_{split}.csv',index=False)
    baseline=next(j for j in JOBS if j['target']==target and j['method']=='erm' and j['arch']=='resnet' and not j['filtered'])
    a=PRED[(tag,'external')]; b=PRED[(baseline['tag'],'external')]
    paired=bootstrap(a,a.p_af.to_numpy(),e['threshold'],b.p_af.to_numpy(),CALIBRATION[baseline['tag']]['threshold'])
    paired['target']=target; paired['contrast']='selected_ensemble_minus_ERM'; PAIRED.append(paired)
results=pd.DataFrame(RESULTS); results.to_csv(OUT/'tables/all_metrics.csv',index=False)
pd.DataFrame(OPERATING_RESULTS).to_csv(OUT/'tables/all_metrics_operating_points.csv',index=False)
pd.concat(CIS,ignore_index=True).to_csv(OUT/'tables/patient_bootstrap_CI.csv',index=False)
pd.concat(PAIRED,ignore_index=True).to_csv(OUT/'tables/paired_bootstrap_differences.csv',index=False)
# These are descriptive across-seed summaries for the same selected ResNet specification.
seed_rows=results[results.tag.isin(sum(SELECTED.values(),[])) & results.split.eq('external')]
seed_rows.groupby('target')[['auroc','auprc','sensitivity','f1']].agg(['mean','std','count']).to_csv(OUT/'tables/seed_variability.csv')
display(results[results.split=='external'])

# ============================================================================
# Notebook cell 25 (zero-based 24): code
# ============================================================================
# Publication-friendly plots and subgroup tables; no tuning happens here.
def savefig(name):
    plt.tight_layout(); plt.savefig(OUT/'figures'/f'{name}.png',dpi=180,bbox_inches='tight')
    plt.savefig(OUT/'figures'/f'{name}.pdf',bbox_inches='tight'); plt.show(); plt.close()
for target in CFG['targets']:
    jobs=[j for j in JOBS if j['target']==target]
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for j in jobs:
        h=pd.read_csv(OUT/f"tables/history_{j['tag']}.csv")
        axes[0].plot(h.epoch,h.loss,label=j['tag'].replace(target+'_',''))
        axes[1].plot(h.epoch,h.val_auc,label=j['tag'].replace(target+'_',''))
    axes[0].set(xlabel='Epoch',ylabel='Training loss'); axes[1].set(xlabel='Epoch',ylabel='Mean source validation AUROC')
    axes[1].legend(fontsize=6); savefig(target+'_training')
    keys=[(j['tag'],'external') for j in jobs]+[(target+'_selected_ensemble','external')]
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    for key in keys:
        d=PRED[key]; label=key[0].replace(target+'_','')
        fpr,tpr,_=roc_curve(d.label,d.p_af); axes[0].plot(fpr,tpr,label=label)
        precision,recall,_=precision_recall_curve(d.label,d.p_af); axes[1].plot(recall,precision,label=label)
        rc=risk_curve(d.label,d.p_af.to_numpy(),d.threshold.iloc[0]); axes[2].plot(rc.coverage,rc.risk,label=label)
    axes[0].plot([0,1],[0,1],'k--',alpha=.4)
    axes[0].set(xlabel='False positive rate',ylabel='Sensitivity'); axes[1].set(xlabel='Recall',ylabel='Precision')
    axes[2].set(xlabel='Coverage',ylabel='Selective error rate'); axes[2].legend(fontsize=5)
    savefig(target+'_roc_pr_risk')
    d=PRED[(target+'_selected_ensemble','external')]
    fig,axes=plt.subplots(1,3,figsize=(13,4))
    for split in ['source_test','external']:
        q=PRED[(target+'_selected_ensemble',split)]
        observed,predicted=calibration_curve(q.label,q.p_af,n_bins=10,strategy='quantile')
        axes[0].plot(predicted,observed,'o-',label=split)
    # Raw vs calibrated uncertainty for the first selected model.
    raw=PRED[(SELECTED[target][0],'external')]
    observed,predicted=calibration_curve(raw.label,raw.p_uncalibrated,n_bins=10,strategy='quantile')
    axes[0].plot(predicted,observed,'--',label='single raw external')
    axes[0].plot([0,1],[0,1],'k:',alpha=.5); axes[0].legend(fontsize=7)
    axes[0].set(xlabel='Predicted AF probability',ylabel='Observed AF fraction')
    cm=confusion_matrix(d.label,d.prediction,labels=[0,1]); axes[1].imshow(cm,cmap='Blues')
    for i in range(2):
        for k in range(2): axes[1].text(k,i,str(cm[i,k]),ha='center',va='center',color='white' if cm[i,k]>cm.max()/2 else 'black')
    axes[1].set(xticks=[0,1],yticks=[0,1],xticklabels=['SR','AF'],yticklabels=['SR','AF'],xlabel='Predicted',ylabel='True')
    for label in [0,1]: axes[2].hist(d.loc[d.label==label,'p_af'],bins=25,alpha=.6,label=['SR','AF'][label])
    axes[2].set(xlabel='AF probability',ylabel='Records',xlim=(0,1)); axes[2].legend(); savefig(target+'_calibration_confusion')
    source_train=ROTATIONS[target].query("split=='train'")
    qcut=source_train.hf_ratio.quantile(.75)
    d=d.copy(); d['quality_proxy']=np.where(d.hf_ratio>qcut,'high_HF_ratio','lower_HF_ratio')
    subs=[]
    for column in ['quality_proxy','sex']:
        for name,g in d.groupby(column):
            if g.label.nunique()==2 and len(g)>=30:
                subs.append(dict(target=target,group=column,value=name,n=len(g),**metrics(g.label,g.p_af,g.threshold.iloc[0])))
    pd.DataFrame(subs).to_csv(OUT/f'tables/subgroups_{target}.csv',index=False)
    d[d.label!=d.prediction].sort_values('entropy').to_csv(OUT/f'tables/confident_errors_{target}.csv',index=False)
    review=[]
    for flag,g in d.groupby('review'):
        review.append(dict(review=bool(flag),n=len(g),error_rate=float((g.label!=g.prediction).mean())))
    pd.DataFrame(review).to_csv(OUT/f'tables/review_groups_{target}.csv',index=False)
    # Examples: no more than 4 physical ECGs displayed, all 12 leads saved in cache.
    example_ids=d[d.label!=d.prediction].head(4).record_id
    if len(example_ids):
        fig,axs=plt.subplots(len(example_ids),1,figsize=(12,2*len(example_ids)),squeeze=False)
        for ax,rid in zip(axs[:,0],example_ids):
            row=manifest[manifest.record_id==rid].iloc[0]
            ax.plot(np.arange(1000)/100,X[int(row.cache_index),1]); ax.set_title(f'{rid} | true {int(row.label)}',fontsize=9)
            ax.set(xlabel='Seconds',ylabel='Lead II (mV)')
        savefig(target+'_error_waveforms')

# ============================================================================
# Notebook cell 26 (zero-based 25): code
# ============================================================================
# Predeclared additive 0.05 mV noise and a 0.2 mV, 0.3 Hz baseline-wander stress.
# Apply to the first selected model; evaluate all external records in bounded batches.
STRESS=[]
for target,tags in SELECTED.items():
    j=next(j for j in JOBS if j['tag']==tags[0]); model,ck=load_model(j['path'])
    f=ROTATIONS[target].query("split=='external'").reset_index(drop=True)
    ds=ECGDataset(f,(ck['mean'].numpy(),ck['std'].numpy()),False)
    for condition in ['noise_0.05mV','wander_0.2mV']:
        seed_all(3407); zz=[]
        with torch.inference_mode():
            for x,_,_ in loader(ds,ck['batch']):
                x=x.to(DEVICE); sd=ck['std'].to(DEVICE)[None,:,None]
                if condition=='noise_0.05mV': x=x+.05*torch.randn_like(x)/sd
                else:
                    wave=.2*torch.sin(2*torch.pi*.3*torch.arange(1000,device=DEVICE)/100)
                    x=x+wave[None,None,:]/sd
                with autocast(): z,_=model(x)
                zz.append(z.float().cpu().numpy())
        c=CALIBRATION[j['tag']]; p=expit(np.concatenate(zz)/c['temperature'])
        STRESS.append(dict(target=target,condition=condition,**metrics(f.label,p,c['threshold'])))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
pd.DataFrame(STRESS).to_csv(OUT/'tables/noise_stress.csv',index=False)

# ============================================================================
# Notebook cell 27 (zero-based 26): markdown
# ============================================================================
# ## 25 · Portable demo model, report, verification and download
# The bundle contains independent selected model weights, model definitions, exactly matched preprocessing, source-fitted normalization, temperatures, classification/review thresholds, a CPU inference API and an optional Streamlit starter. The demo accepts a numeric NumPy array of shape `[12, 1000]` (100 Hz) or `[12, 5000]` (500 Hz), in **mV**, with lead order `I, II, III, aVR, aVL, aVF, V1…V6`. Arbitrary images and single-lead wearables are unsupported. No demo input is transmitted to an external service.
#
# An exported-model round-trip check compares probabilities against in-notebook inference. The final ZIP includes all research outputs and model assets, but not raw datasets or large caches. Browser download starts automatically; a link and a Drive copy provide alternatives.

# ============================================================================
# Notebook cell 28 (zero-based 27): code
# ============================================================================
INFERENCE_SOURCE="from pathlib import Path\nimport json\nimport numpy as np\nimport torch\nimport wfdb\nfrom scipy.signal import resample_poly, butter, sosfiltfilt\nfrom model import make_model\nLEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6']\ndef physical_ecg(path):\n    rec=wfdb.rdrecord(str(path),physical=True)\n    names=[s.upper() for s in rec.sig_name]\n    assert len(names)==12 and set(names)==set(LEADS), 'missing_or_duplicate_leads'\n    assert abs(rec.sig_len/rec.fs-10)<1e-6, 'wrong_duration'\n    order=[names.index(s) for s in LEADS]\n    x=np.asarray(rec.p_signal[:,order],dtype=np.float64).T\n    factors={'mv':1.,'uv':.001,'µv':.001,'μv':.001,'v':1000.}\n    for j,k in enumerate(order):\n        unit=rec.units[k].lower()\n        assert unit in factors, f'unknown_unit:{unit}'\n        x[j]*=factors[unit]\n    assert np.isfinite(x).all(), 'nonfinite'\n    assert np.all(np.ptp(x,axis=1)>1e-5), 'flat_lead'\n    assert np.max(np.abs(x))<100, 'implausible_amplitude_check_units'\n    return x, float(rec.fs)\ndef process_signal(x,fs,filtered=False):\n    from fractions import Fraction\n    ratio=Fraction(100/float(fs)).limit_denominator(10000)\n    x=resample_poly(x,ratio.numerator,ratio.denominator,axis=-1) if fs!=100 else x.copy()\n    assert x.shape==(12,1000) and np.isfinite(x).all(), 'bad_processed_shape'\n    if filtered: x=sosfiltfilt(butter(3,[.5,40],btype='bandpass',fs=100,output='sos'),x,axis=-1)\n    return np.ascontiguousarray(x,dtype=np.float32)\n\nclass Predictor:\n    def __init__(self,folder):\n        self.folder=Path(folder)\n        self.meta=json.loads((self.folder/'bundle.json').read_text())\n        self.members=[]\n        for item in self.meta['members']:\n            ck=torch.load(self.folder/item['file'],map_location='cpu',weights_only=True)\n            m=make_model(ck['arch']); m.load_state_dict(ck['state']); m.eval()\n            self.members.append((m,ck,item['temperature']))\n    @torch.inference_mode()\n    def predict_array(self,x,fs=100,leads=LEADS,units='mV'):\n        x=np.asarray(x,dtype=np.float64)\n        names=[s.upper() for s in leads]\n        if x.ndim!=2 or x.shape[0]!=12 or len(names)!=12 or set(names)!=set(LEADS):\n            raise ValueError('Expected 12 unique canonical leads, [leads, samples].')\n        if fs not in [100,500] or x.shape[1]!=int(fs*10): raise ValueError('Need 10 seconds at 100 or 500 Hz.')\n        if units!='mV': raise ValueError('Convert voltages to mV before inference.')\n        x=x[[names.index(k) for k in LEADS]]\n        if not np.isfinite(x).all() or np.any(np.ptp(x,axis=1)<=1e-5) or np.max(np.abs(x))>=100:\n            raise ValueError('Nonfinite, flat or implausible physical signal.')\n        pp=[]\n        for m,ck,temp in self.members:\n            a=process_signal(x,fs,ck['filtered'])\n            a=(torch.from_numpy(a)-ck['mean'][:,None])/ck['std'][:,None]\n            logits,_=m(a[None]); pp.append(float(torch.sigmoid(logits/temp)[0]))\n        p=float(np.mean(pp)); q=np.clip(p,1e-7,1-1e-7)\n        ent=float(-q*np.log(q)-(1-q)*np.log(1-q))\n        return dict(p_af=p,label='AF' if p>=self.meta['threshold'] else 'SR',\n            entropy=ent,variance=float(np.var(pp)),review=bool(ent>self.meta['review_entropy']))\n    def predict_wfdb(self,path):\n        x,fs=physical_ecg(path)\n        return self.predict_array(x,fs)\n"
APP_SOURCE='from pathlib import Path\nimport numpy as np\nimport streamlit as st\nfrom inference import Predictor\nst.title(\'Research ECG rhythm demo\')\nst.caption(\'AF versus sinus rhythm only. Not for medical decisions.\')\n@st.cache_resource\ndef load(): return Predictor(Path(__file__).parent)\npredictor=load()\nst.write(\'Upload 10 seconds of 12-lead numerical ECG in mV: I, II, III, aVR, aVL, aVF, V1–V6.\')\nf=st.file_uploader(\'NumPy ECG array (.npy)\',type=[\'npy\'])\nfs=st.selectbox(\'Sampling frequency (Hz)\',[100,500])\nif f:\n    if f.size>2_000_000:\n        st.error(\'File too large for a 10-second ECG.\')\n    else:\n        try:\n            x=np.load(f,allow_pickle=False)\n            r=predictor.predict_array(x,fs)\n            st.metric(\'Estimated AF probability\',f"{r[\'p_af\']:.1%}")\n            st.write(\'Model label:\',r[\'label\'])\n            if r[\'review\']: st.warning(\'High uncertainty: review flagged.\')\n            st.line_chart(x[1,::max(1,fs//100)])\n        except Exception as e: st.error(str(e))\n'

# ============================================================================
# Notebook cell 29 (zero-based 28): code
# ============================================================================
# The primary Georgia-held-out model is exported; selection is exclusively source-based.
export_target=CFG['targets'][0]; e=ENSEMBLE[export_target]
BUNDLE=OUT/'bundle'
(BUNDLE/'inference.py').write_text(INFERENCE_SOURCE)
(BUNDLE/'streamlit_app.py').write_text(APP_SOURCE)
members=[]
for i,tag in enumerate(e['tags']):
    job=next(j for j in JOBS if j['tag']==tag); name=f'member_{i}.pt'
    shutil.copy2(job['path'],BUNDLE/name)
    members.append(dict(file=name,temperature=CALIBRATION[tag]['temperature'],tag=tag))
write_json(BUNDLE/'bundle.json',dict(members=members,threshold=e['threshold'],review_entropy=e['review_entropy'],
    external_domain=export_target,leads=LEADS,fs=100,units='mV',samples=1000,
    operating_point=CFG['operating_point'],threshold_youden=e['threshold_youden'],
    source_operating_diagnostics=e['source_operating_diagnostics'],
    intended_use='Research AF-vs-explicit-SR demonstration only',protocol_sha256=FREEZE_HASH))
(BUNDLE/'requirements.txt').write_text('torch>=2.3,<3\nnumpy>=1.26,<3\nscipy>=1.11,<2\nwfdb==4.3.0\nstreamlit>=1.35,<2\n')
(BUNDLE/'README.md').write_text('Run `pip install -r requirements.txt` then `streamlit run streamlit_app.py`. '
    'Python API: `from inference import Predictor; p=Predictor("."); p.predict_array(x,fs=100)`. '
    'The array must be 12×1000 in mV at 100 Hz, in the documented lead order. '
    'See bundle.json and the research report for restrictions. Never use a low uncertainty score as a clinical safety guarantee.')
# Verify a saved/reloaded CPU bundle against original checkpoint FP32 inference.
import importlib.util
sys.path.insert(0,str(BUNDLE))
spec=importlib.util.spec_from_file_location('exported_inference',BUNDLE/'inference.py')
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
predictor=module.Predictor(BUNDLE)
example=ROTATIONS[export_target].query("split=='source_test'").iloc[0]
x=np.array(X[int(example.cache_index)],copy=True)
exported=predictor.predict_array(x,100)
expected=[]
for member in members:
    ck=torch.load(BUNDLE/member['file'],map_location='cpu',weights_only=True)
    m=make_model(ck['arch']).eval()
    m.load_state_dict(ck['state'])
    xx=(torch.from_numpy(x)-ck['mean'][:,None])/ck['std'][:,None]
    with torch.inference_mode(): z,_=m(xx[None]); expected.append(float(torch.sigmoid(z/member['temperature'])[0]))
assert abs(exported['p_af']-np.mean(expected))<1e-6, 'Export round-trip mismatch'
# Negative input checks: malformed shape, NaNs and flat channels must be rejected.
for bad in [np.zeros((1,1000)),np.full((12,1000),np.nan),np.zeros((12,1000))]:
    try: predictor.predict_array(bad,100)
    except ValueError: pass
    else: raise AssertionError('Invalid demo input accepted')
write_json(OUT/'export_validation.json',dict(round_trip_passed=True,p_af=exported['p_af'],invalid_input_checks=3))
del predictor,m; gc.collect()

# ============================================================================
# Notebook cell 30 (zero-based 29): code
# ============================================================================
def regenerate_metric_table(prediction_dir):
    rows=[]
    for path in sorted(Path(prediction_dir).glob('*.csv')):
        d=pd.read_csv(path)
        for domain,g in d.groupby('domain'):
            rows.append(dict(file=path.name,domain=domain,**metrics(g.label,g.p_af,g.threshold.iloc[0])))
    return pd.DataFrame(rows)
regenerated=regenerate_metric_table(OUT/'predictions')
regenerated.to_csv(OUT/'tables/regenerated_metrics.csv',index=False)
assert len(regenerated)==len(results)
for r in results.itertuples():
    got=regenerated.query('file==@name and domain==@domain',local_dict={'name':r.tag+'_'+r.split+'.csv','domain':r.domain})
    assert len(got)==1 and np.isclose(got.auroc.iloc[0],r.auroc,atol=1e-9)
summary=results[results.split=='external'].to_csv(index=False)
report=f"""# Cross-domain AF versus explicit SR experiment
Mode: {MODE}. Completed fits: {len(JOBS)}. Skipped jobs: {len(SKIPPED)}.
Elapsed minutes through analysis: {(time.monotonic()-START)/60:.1f}.
Protocol SHA256: {FREEZE_HASH}.

## Protocol and interpretation
Source validation chose checkpoints and alignment weights. Source calibration fitted temperatures, the original Youden thresholds, and sensitivity-oriented thresholds (both saved). The active operating point is {CFG['operating_point']}; source sensitivity goal {CFG['source_target_sensitivity']:.0%} subject to a source specificity floor {CFG['source_min_specificity']:.0%} per domain. Source thresholds cannot guarantee external performance.
No external labels are accessed by the threshold-selection code. This revision was developed after inspecting previous Georgia results: any repeat Georgia evaluation is exploratory, not a fresh external confirmation. No claims of improvement, novelty or clinical reliability are assumed.
PTB-XL is a dataset domain; Shaoxing and Georgia record IDs are unverified patient proxies.
Cross-hospital patient identity cannot be independently linked. Exact duplicates were removed, but near duplicates may remain.
Strict SR is not healthy-control status; non-rhythm comorbidities are allowed. Ineligible and invalid signals are logged.
The pilot uses all eligible source-training records, three seeds, a five-minute per-fit cap and evaluates only Georgia.
Full mode executes all three predeclared rotations and three selected-model seeds, but may take many hours.
Time-capped fits can be undertrained. Domain/class-balanced sampling changes training priors; source calibration
may not correct unseen-domain prior shift. Confidence intervals condition on the fitted models.
Do not claim negative paired CI endpoints or small point differences prove superiority; inspect all measures and seeds.
A subgroup high-frequency ratio is a proxy, not an expert quality annotation. Synthetic stress is not real-device validation.
The demo ensemble uses source-selected ResNets; compare against InceptionTime and CORAL even if they outperform it. Compare both operating points using tables/all_metrics_operating_points.csv, and inspect the source calibration constraints file.

## External results (CSV)
{summary}
## Reproducibility
All predictions, freeze metadata, seeds, exclusions, split manifests, training histories, versions,
portable weights, confidence intervals and regenerated metrics are included. Raw datasets/caches are excluded.
Dataset download locations and hashes appear in download_provenance.json.

## Skipped jobs
{json.dumps(SKIPPED)}

## Sources
- PTB-XL v1.0.3: https://physionet.org/content/ptb-xl/1.0.3/
- Challenge hospital membership and published checksums: https://physionet.org/content/challenge-2021/1.0.3/
- Challenge cohort/patient counts: https://physionet.org/files/challenge-2021/1.0.2/papers/2021ChallengePaperPMEA.pdf
- Georgia 12-lead ECGs (Challenge 2021 v1.0.3): https://physionet.org/content/challenge-2021/1.0.3/
- ECG-Arrhythmia v1.0.0 waveforms and annotations: https://physionet.org/content/ecg-arrhythmia/1.0.0/
- Strodthoff et al., PTB-XL benchmarks: https://doi.org/10.1109/JBHI.2020.3022989
- Deep CORAL: https://arxiv.org/abs/1607.01719
- Temperature scaling: https://proceedings.mlr.press/v70/guo17a.html
- InceptionTime: https://doi.org/10.1007/s10618-020-00710-y

Please cite the datasets and method papers, observe their licenses, and report pilot limitations.
"""
(OUT/'research_report.md').write_text(report)
# Include runnable notebook source when Colab can supply it; absence is explicitly logged.
try:
    from google.colab import _message
    response=_message.blocking_request('get_ipynb',timeout_sec=20)
    write_json(OUT/'executed_notebook.ipynb',response['ipynb'])
except Exception as exc:
    write_json(OUT/'notebook_capture_note.json',dict(message='Save the executed notebook manually from Colab File menu.',reason=str(exc)))
write_json(OUT/'artifact_hashes.json',{str(p.relative_to(OUT)):digest(p) for p in OUT.rglob('*') if p.is_file() and p.name!='artifact_hashes.json'})
sync_outputs()
ZIP=Path(shutil.make_archive(str(OUT.parent/f'ecg_results_{RUN}'),'zip',OUT))
with zipfile.ZipFile(ZIP) as z: assert z.testzip() is None
if DRIVE is not None: shutil.copy2(ZIP,DRIVE/ZIP.name)
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))
print('Verified result archive:',ZIP,'size MiB:',round(ZIP.stat().st_size/2**20,1))
if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(ZIP))
    except ImportError: print('Outside Colab: use the result archive link above.')

# ============================================================================
# Notebook cell 31 (zero-based 30): markdown
# ============================================================================
# ### Validation and limits
# Earlier versions were validated on a synthetic CPU fixture; the present exploratory revision has been syntax checked and its source-threshold routine unit tested, but NOT run end-to-end on ECG waveforms in this environment. WFDB unit conversion and anti-alias resampling were also checked on a real PhysioNet AF recording. Download URLs were checked for HTTP range access, and a sample of official Challenge waveform hashes was matched against ECG-Arrhythmia. A complete PhysioNet download, full cohort run, CUDA memory probe and T4 runtime benchmark must be performed in Colab. Runtime assertions and checkpoints are included for those stages. A passing synthetic test establishes implementation behavior, not ECG accuracy or clinical validity.
#
# This is an explicitly exploratory revision after review of a previous Georgia result. For a confirmatory performance claim, use a newly independent unseen patient cohort with the protocol frozen in advance. To run the complete research plan, change `MODE='full'` before running any cells.
